# Clean CO₂-domain case-disjoint benchmark and OOF field predictions

This notebook is a self-contained, resumable Kaggle T4 ×2 run. It reads the filtered CO₂-candidate parquet when available; otherwise it loads the combined source table and applies the documented fallback `density <= 950 kg m⁻³`. The `T_205` case keeps its ID while its input feature is set to 305 K. A zone-only CFD-Post re-export is preferred and is required to confirm fluid identity.

The 17-model suite comprises the original pointwise, tree, neural-operator, graph, transformer, mesh-aligned, and regular-grid adaptations. The run also includes FiLM-versus-concatenation. A proposed loop-unrolled level/shape architecture prototype is provided separately, but remains pending registration and training until validated loop-path features and EOS inputs are available. Four configuration/geometry group holdouts are retained; mesh-aligned-only methods are explicitly marked not applicable to unseen groups. Per-case OOF predictions are saved for scheduled models. Seeds 2026, 2027, and 2028 write to separate subfolders under `/kaggle/working`. Fold/model checkpoints, progress logs, and a wall-clock guard support resumption; the status table distinguishes completed, pending, and not-applicable experiments.


# Clean-data case-disjoint benchmark

## Data and scope

- The loader accepts a combined tabular Parquet or CSV table, normalizes local snake-case and common Fluent/Kaggle unit-bearing headers, and fails on missing or inconsistent case metadata.
- Operating inputs are read from columns when available and otherwise parsed from the case identifier. Loop extents are derived from retained CO₂-candidate nodes. The `T_205` feature is set to 305 K based on the raw nodal-temperature audit while the original case ID is preserved.
- Nodes with density above 950 kg m⁻³ are excluded as an EOS-informed secondary-fluid candidate filter. This is not a confirmed Fluent zone classification; check the zone IDs before interpreting the filtered table as definitively CO₂-only.
- The outputs are sampled on cross-sectional planes, not a complete volumetric mesh. No wall temperature or heat-transfer coefficient is available.

## Evaluation protocol

- Five-fold case-level cross-validation holds out each CFD case exactly once. No nodes from a test case are used for training or neural validation.
- A separate four-group leave-one-configuration/geometry-group-out protocol tests models that accept pooled coordinates and operating conditions. Nearest-case and POD transfer are mesh-aligned and are marked not applicable when their training group is absent.
- Neural checkpoint selection uses case-level validation data only. Exact case memberships, per-case metrics, run status, and training history are saved.
- F-FNO and U-FNO3D adaptations are retained and reported as regular-grid remapping models; the lattice-oracle output is a remapping diagnostic, not a predictive model.
- The FiLM-MLP and concatenation-conditioned MLP use the same coordinate features, hidden width/depth, loss, seed, and training budget. This is a conditioning-mechanism ablation, not a parameter-count-matched comparison.

## Reproducibility and outputs

- The default seed is 2026; set `SCWR_SEED` to 2027 or 2028 for independent runs. Each seed writes to its own output directory.
- Task checkpoints, progress logs, validation-based checkpoint selection, and an 11-hour wall-clock guard make the Kaggle run resumable.
- OOF predictions, case-level truth arrays, selection tables, maps, exact splits, and metrics are saved under the seed-specific output directory.
- Legacy all-exported-node scores remain separate from clean-domain results and are labeled as contamination-sensitivity references.


In [ ]:
# Imports and configuration
import os, gc, json, math, random, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import scipy
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.spatial import cKDTree
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import matplotlib
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except Exception:
    def display(x): print(x)
warnings.filterwarnings("ignore")

SMOKE = os.environ.get("SCWR_SMOKE", "0") == "1"      # tiny settings for code testing only
SEED = int(os.environ.get("SCWR_SEED", "2026"))
N_FOLDS = 2 if SMOKE else 5
N_VAL_CASES = 3 if SMOKE else 4
MAX_TOTAL_HOURS = 0.5 if SMOKE else 11.0                # hard guard, leaves margin under Kaggle's 12 h
WALL_START = time.monotonic()
DEADLINE = WALL_START + MAX_TOTAL_HOURS * 3600
BOOTSTRAP = 300 if SMOKE else 2000

# ---- equal budget for all neural models --------------------------------------------------
SAMPLE_BUDGET = 40_000 if SMOKE else 80_000_000        # supervised node samples per model per split
BATCH = 256 if SMOKE else 8192
LR = 5e-4
EVAL_CHUNK = 4096 if SMOKE else 32768
VAL_NODES_PER_CASE = 400 if SMOKE else 30000
TS_SAMPLE = 2000 if SMOKE else 50000                    # nodes/case used to fit target scalers
MEM_DIV = 1                                             # doubled automatically after a CUDA OOM retry
CAP = {                                                 # wall-clock cap (s) per model per split
    "pointwise": 12 if SMOKE else 11 * 60,
    "transolver": 12 if SMOKE else 12 * 60,
    "gams": 12 if SMOKE else 14 * 60,
    "fno": 12 if SMOKE else 8 * 60,
}
# ---- architecture sizes -------------------------------------------------------------------
NN_WIDTH, NN_DEPTH, NN_LATENT = (32, 2, 16) if SMOKE else (256, 4, 128)
PHYS_W, PHYS_BATCH = 0.05, (128 if SMOKE else 1024)
TRANS_WIDTH, TRANS_BLOCKS, TRANS_SLICES = (32, 2, 8) if SMOKE else (128, 4, 32)
TRANS_CTX, TRANS_QRY = (256, 128) if SMOKE else (8192, 4096)
GAMS_WIDTH, GAMS_BLOCKS, GAMS_NODES, GAMS_K = (24, 2, 256, 6) if SMOKE else (96, 4, 8192, 6)
GAMS_BINS1, GAMS_BINS2 = ((6, 6, 3), (3, 3, 2)) if SMOKE else ((20, 20, 6), (10, 10, 3))
LAT = (16, 16, 8) if SMOKE else (64, 64, 16)           # F-FNO / U-FNO lattice on normalized coordinates
FNO_WIDTH, FNO_MODES, FNO_STEPS, FNO_BATCH = (8, 4, 40, 2) if SMOKE else (24, 12, 2000, 2)
TREE_NODES_PER_CASE = 400 if SMOKE else 30000
RF_TREES, ET_TREES = (6, 6) if SMOKE else (48, 64)
TREE_MAX_LEAF, TREE_MIN_LEAF, TREE_BOOT = (64, 5, 0.5) if SMOKE else (4096, 5, 0.3)
XGB_TREES, CAT_ITERS = (10, 10) if SMOKE else (300, 350)
N_CPU = max(1, min(4, os.cpu_count() or 1))
POD_MAX_MODES, POD_ENERGY = 12, 0.9999
RBF_LENGTH_SCALES = [0.25, 0.5, 1., 2., 4., 8.]
RBF_ALPHAS = [1e-6, 1e-4, 1e-2, 1e-1, 1.]

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

seed_all(SEED)
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3 if N_GPUS else 0
if N_GPUS and VRAM_GB < 8:
    BATCH, EVAL_CHUNK, GAMS_NODES = 2048, 8192, 4096
print(f"PyTorch {torch.__version__} | device={DEVICE} | GPUs={N_GPUS} | VRAM/GPU={VRAM_GB:.1f} GB | CPUs={N_CPU}")
print(f"SMOKE={SMOKE} | budget={SAMPLE_BUDGET:,} samples/model/split | batch={BATCH} | deadline={MAX_TOTAL_HOURS} h")

def time_remaining():
    return max(0.0, DEADLINE - time.monotonic())

def ensure_time(label, need=60):
    if time_remaining() <= need:
        raise TimeoutError(f"time budget reached before {label}")

# ---- paths ---------------------------------------------------------------------------------
PARQUET_PATH = os.environ.get("SCWR_PARQUET", "/kaggle/input/datasets/barghavdhamodharan/scwr-data/mega_dataset.parquet")
if not Path(PARQUET_PATH).exists():
    candidates = sorted(Path("/kaggle/input").rglob("mega_dataset.parquet")) if Path("/kaggle/input").exists() else []
    if not candidates and Path("/kaggle/input").exists():
        candidates = sorted(Path("/kaggle/input").rglob("mega_dataset.csv"))
    if not candidates and (Path.cwd() / "mega_dataset.parquet").exists():
        candidates = [Path.cwd() / "mega_dataset.parquet"]
    if len(candidates) != 1:
        raise FileNotFoundError(f"Expected exactly one tabular mega_dataset.parquet or mega_dataset.csv; found {len(candidates)}. Set SCWR_PARQUET.")
    PARQUET_PATH = str(candidates[0])
OUTPUT_ROOT = Path(os.environ.get("SCWR_OUT", "/kaggle/working" if Path("/kaggle/working").exists() else "mega_model_outputs"))
OUTPUT_DIR = OUTPUT_ROOT / f"seed_{SEED}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR = OUTPUT_DIR / "case_predictions"; PRED_DIR.mkdir(exist_ok=True)
PRED_DATA_DIR = OUTPUT_DIR / "case_data"; PRED_DATA_DIR.mkdir(exist_ok=True)
PRED_MODELS = {"Mean baseline", "Nearest case", "Random Forest", "Extra Trees", "XGBoost", "CatBoost", "DeepONet", "PINN data-only", "PINN continuity", "Domain-informed FiLM-MLP", "Domain-informed Concat-MLP", "Latent Neural Operator", "Transolver-style", "F-FNO", "U-FNO3D", "GAMS-GNN-style", "POD-RBF", "POD-GPR", "Loop-Unrolled Offset-Shape Operator"}
print("Parquet:", PARQUET_PATH, "| output:", OUTPUT_DIR)

TARGETS = ["density_kg_m3", "pressure_pa", "temperature_c", "turbulence_eddy_dissipation_m2_s3",
           "turbulence_kinetic_energy_m2_s2", "velocity_u_m_s", "velocity_v_m_s", "velocity_w_m_s"]
HEAVY_TAILED = {"turbulence_eddy_dissipation_m2_s3", "turbulence_kinetic_energy_m2_s2"}
NUMERIC_CASE = ["power_parameter_value", "pressure_bar", "mass_flow_kg_s", "temperature_parameter_value",
                "loop_x_extent_m", "loop_y_extent_m"]
COORDS = ["x_m", "y_m", "z_m"]
META = ["case_id", "configuration", "geometry_class", *NUMERIC_CASE]
NT = len(TARGETS)
assert TARGETS.index("density_kg_m3") == 0 and TARGETS[5:8] == ["velocity_u_m_s", "velocity_v_m_s", "velocity_w_m_s"]


## Data loading, case table, features, and splits
Only case-level metadata is read first. Node arrays are held as float32 views per case. Coordinates are normalized **per case** to [0, 1]^3, with loop extents supplied as case features, so all loop sizes and orientations can be pooled.

In [ ]:
# Normalize Kaggle and local mega-dataset schemas before building case arrays.
t_load = time.monotonic()
import re

def _key(name):
    return re.sub(r"[^a-z0-9]", "", str(name).lower())

_full = (
    pq.read_table(PARQUET_PATH).to_pandas()
    if PARQUET_PATH.lower().endswith(".parquet")
    else pd.read_csv(PARQUET_PATH)
)
_lookup = {_key(column): column for column in _full.columns}
_aliases = {
    "case_id": ["case_id", "case id", "case", "case_name", "filename", "source_filename"],
    "configuration": ["configuration", "config", "orientation"],
    "geometry_class": ["geometry_class", "geometry", "loop_size"],
    "power_parameter_value": ["power_parameter_value", "power_w", "heater_power_w", "power", "q_w"],
    "pressure_bar": ["pressure_bar", "operating_pressure_bar", "p_op_bar", "pressure_input_bar"],
    "mass_flow_kg_s": ["mass_flow_kg_s", "massflow_kg_s", "mdot_kg_s", "coolant_inlet_mass_flow_kg_s"],
    "temperature_parameter_value": ["temperature_parameter_value", "temperature_input_k", "temperature_k", "t_in_k", "coolant_inlet_temperature_k"],
    "loop_x_extent_m": ["loop_x_extent_m", "x_extent_m", "loop_length_x_m"],
    "loop_y_extent_m": ["loop_y_extent_m", "y_extent_m", "loop_length_y_m"],
    "x_m": ["x_m", "x", "x_meters", "x_m_"],
    "y_m": ["y_m", "y", "y_meters", "y_m_"],
    "z_m": ["z_m", "z", "z_meters", "z_m_"],
    "density_kg_m3": ["density_kg_m3", "density", "density_kg_m3_"],
    "pressure_pa": ["pressure_pa", "pressure", "gauge_pressure_pa", "pressure_pa_"],
    "temperature_c": ["temperature_c", "temperature", "temperature_c_"],
    "turbulence_eddy_dissipation_m2_s3": ["turbulence_eddy_dissipation_m2_s3", "turbulence_eddy_dissipation", "eddy_dissipation", "turbulenceeddydissipationm2s3"],
    "turbulence_kinetic_energy_m2_s2": ["turbulence_kinetic_energy_m2_s2", "turbulence_kinetic_energy", "tke"],
    "velocity_u_m_s": ["velocity_u_m_s", "velocity_u", "velocity u m s^-1", "u_velocity", "u"],
    "velocity_v_m_s": ["velocity_v_m_s", "velocity_v", "velocity v m s^-1", "v_velocity", "v"],
    "velocity_w_m_s": ["velocity_w_m_s", "velocity_w", "velocity w m s^-1", "w_velocity", "w"],
}
_rename = {}
for _canonical, _candidates in _aliases.items():
    if _canonical not in _full.columns:
        _source = next((_lookup.get(_key(candidate)) for candidate in _candidates if _lookup.get(_key(candidate)) is not None), None)
        if _source is not None:
            _rename[_source] = _canonical
_full = _full.rename(columns=_rename)
if "case_id" not in _full.columns:
    raise ValueError(f"Case identifier/filename missing. Dataset columns: {list(_full.columns)}")
_full["case_id"] = _full["case_id"].astype(str).str.replace(r"\.csv$", "", regex=True)
_parsed = _full["case_id"].str.extract(
    r"^(?P<cfg>[A-Za-z]+)_P(?P<power>\d+(?:\.\d+)?)_(?P<pbar>\d+(?:\.\d+)?)_bar_"
    r"(?P<mdot>\d+(?:\.\d+)?)kgs_T_(?P<tin>-?\d+(?:\.\d+)?)$"
)
for _name, _parsed_name in [
    ("configuration", "cfg"),
    ("power_parameter_value", "power"),
    ("pressure_bar", "pbar"),
    ("mass_flow_kg_s", "mdot"),
    ("temperature_parameter_value", "tin"),
]:
    if _name not in _full.columns:
        _full[_name] = _parsed[_parsed_name]
    else:
        _full[_name] = _full[_name].fillna(_parsed[_parsed_name])
for _name in ["power_parameter_value", "pressure_bar", "mass_flow_kg_s", "temperature_parameter_value"]:
    _full[_name] = pd.to_numeric(_full[_name], errors="raise")
_required = ["case_id", "configuration", *COORDS, *TARGETS, *NUMERIC_CASE[:4]]
_missing = sorted(set(_required) - set(_full.columns))
if _missing:
    raise ValueError(f"Missing canonical fields {_missing}. Dataset columns: {list(_full.columns)}")
if _full["case_id"].isna().any():
    raise ValueError("Rows without a case identifier cannot be assigned to a case-level split.")
_full.loc[_full["case_id"].str.endswith("_T_205"), "temperature_parameter_value"] = 305.0
_case_metadata = ["configuration", *NUMERIC_CASE[:4]]
_metadata_variation = _full.groupby("case_id", observed=True)[_case_metadata].nunique(dropna=False)
if _metadata_variation.gt(1).any().any():
    raise ValueError("Operating metadata varies within at least one case.")
_candidate_nodes = _full.loc[
    _full["density_kg_m3"] <= 950.0, ["case_id", *COORDS, *TARGETS]
].copy()
_candidate_count = len(_candidate_nodes)
_nodes = _candidate_nodes.dropna(subset=["case_id", *COORDS, *TARGETS]).copy()
print(f"Removed {_candidate_count - len(_nodes):,} candidate rows with null case/coordinate/target values.")
if _nodes.empty:
    raise ValueError("The CO2-candidate filter removed all nodes.")
_extents = _nodes.groupby("case_id", sort=True).agg(
    x_min=("x_m", "min"), x_max=("x_m", "max"), y_min=("y_m", "min"), y_max=("y_m", "max")
)
_extents["loop_x_extent_m"] = _extents["x_max"] - _extents["x_min"]
_extents["loop_y_extent_m"] = _extents["y_max"] - _extents["y_min"]
_extents["geometry_class"] = np.where(
    np.maximum(_extents["loop_x_extent_m"], _extents["loop_y_extent_m"]) > 2.0, "large", "small"
)
_meta = _full.drop_duplicates("case_id").drop(
    columns=["geometry_class", "loop_x_extent_m", "loop_y_extent_m"], errors="ignore"
)
_meta = _meta.merge(
    _extents[["loop_x_extent_m", "loop_y_extent_m", "geometry_class"]],
    left_on="case_id", right_index=True, how="left", validate="one_to_one"
)
_meta["configuration"] = _meta["configuration"].astype(str)
case_table = _meta[META].sort_values("case_id").reset_index(drop=True)
if case_table["case_id"].duplicated().any():
    raise ValueError("A case ID has inconsistent operating metadata.")
case_table["mesh_group"] = case_table.configuration + "|" + case_table.geometry_class.astype(str)
CASE_IDS = case_table.case_id.tolist()
CASE_POS = {c: i for i, c in enumerate(CASE_IDS)}
GROUP_OF = dict(zip(case_table.case_id, case_table.mesh_group))
MESH_GROUPS = sorted(case_table.mesh_group.unique())
GROUP_ALIGNED = {g: True for g in MESH_GROUPS}
del _full, _meta, _parsed, _extents, _candidate_nodes
gc.collect()
print(f"CO2-candidate nodes after secondary-fluid and null filters: {len(_nodes):,}")
_nodes["case_id"] = pd.Categorical(_nodes["case_id"], categories=CASE_IDS, ordered=True)
_codes = _nodes["case_id"].cat.codes.to_numpy()
_cats = np.asarray(CASE_IDS)
_order = np.argsort(_codes, kind="stable")
_coords = _nodes[COORDS].to_numpy(np.float32)[_order]
_y = _nodes[TARGETS].to_numpy(np.float32)[_order]
_counts = np.bincount(_codes, minlength=len(_cats))
del _nodes, _codes, _order
gc.collect()
if not (np.isfinite(_coords).all() and np.isfinite(_y).all()):
    raise ValueError("Non-finite coordinates or targets remain after data normalization.")
RAW_C, RAW_Y, NORM, SPAN = {}, {}, {}, {}
_ends = np.cumsum(_counts); _starts = _ends - _counts
for i, cid in enumerate(_cats):
    c = _coords[_starts[i]:_ends[i]]
    RAW_C[cid] = c
    RAW_Y[cid] = _y[_starts[i]:_ends[i]]
    low, span = c.min(0), np.maximum(np.ptp(c, axis=0), 1e-6)
    NORM[cid] = ((c - low) / span).astype(np.float32)
    SPAN[cid] = span.astype(np.float32)
missing = set(CASE_IDS) - set(RAW_Y)
if missing:
    raise ValueError(f"Cases without retained CO2-candidate nodes: {sorted(missing)[:5]}")
case_table["n_nodes"] = [len(RAW_Y[c]) for c in CASE_IDS]
print(f"Loaded {len(CASE_IDS)} cases and {int(case_table.n_nodes.sum()):,} candidate nodes.")
display(case_table.groupby(["mesh_group"]).agg(cases=("case_id", "size"), nodes_per_case=("n_nodes", "mean")))


In [ ]:
# Case features (pooled models see all of these)
def _orient(cfg, k):
    return float(len(cfg) > k and cfg[k] == "V")
_cfg = pd.get_dummies(case_table.configuration, dtype=float)
_geo = pd.get_dummies(case_table.geometry_class, dtype=float)
_flags = np.array([[_orient(c, 0), _orient(c, 2)] for c in case_table.configuration], dtype=np.float32)  # vertical heater, vertical cooler
CASE_FEAT_NAMES = NUMERIC_CASE + list(_cfg.columns) + list(_geo.columns) + ["vertical_heater", "vertical_cooler"]
CASE_FEAT = np.concatenate([case_table[NUMERIC_CASE].to_numpy(np.float32), _cfg.to_numpy(np.float32),
                            _geo.to_numpy(np.float32), _flags], axis=1)
NF = CASE_FEAT.shape[1]
print("Case features:", CASE_FEAT_NAMES)

In [ ]:
# Splits: stratified k-fold over cases + leave-one-mesh-group-out
def stratified_folds(table, n_folds, seed):
    rng = np.random.default_rng(seed); fold_of = {}; offset = 0
    for g in sorted(table.mesh_group.unique()):
        ids = sorted(table.loc[table.mesh_group == g, "case_id"]); rng.shuffle(ids)
        for k, cid in enumerate(ids):
            fold_of[cid] = (offset + k) % n_folds
        offset += len(ids)
    return fold_of

def pick_val(train_ids, n_val, seed):
    """Validation cases are taken only from groups with >= 8 training cases so small groups are not starved."""
    rng = np.random.default_rng(seed)
    by_g = {}
    for c in train_ids:
        by_g.setdefault(GROUP_OF[c], []).append(c)
    elig = sorted([g for g, v in by_g.items() if len(v) >= 8]) or [max(by_g, key=lambda g: len(by_g[g]))]
    for g in elig:
        rng.shuffle(by_g[g])
    val, k = [], 0
    while len(val) < n_val and any(by_g[g] for g in elig):
        g = elig[k % len(elig)]
        if by_g[g]:
            val.append(by_g[g].pop())
        k += 1
    return sorted(val)

SPLITS = {"kfold": [], "geometry_holdout": []}
_fold_of = stratified_folds(case_table, N_FOLDS, SEED)
for f in range(N_FOLDS):
    test = sorted(c for c in CASE_IDS if _fold_of[c] == f)
    rest = [c for c in CASE_IDS if c not in set(test)]
    val = pick_val(rest, N_VAL_CASES, SEED + f)
    SPLITS["kfold"].append(dict(protocol="kfold", id=f"fold{f+1}", test_ids=test, val_ids=val,
                                train_ids=sorted(set(rest) - set(val))))
for g in MESH_GROUPS:
    test = sorted(case_table.loc[case_table.mesh_group == g, "case_id"])
    rest = [c for c in CASE_IDS if c not in set(test)]
    if len(rest) < 15:
        continue
    val = pick_val(rest, N_VAL_CASES, SEED + 99)
    SPLITS["geometry_holdout"].append(dict(protocol="geometry_holdout", id=f"leave_out:{g}", test_ids=test,
                                           val_ids=val, train_ids=sorted(set(rest) - set(val))))
if SMOKE:
    pass  # Retain all four observed configuration/geometry group holdouts.
for p, sl in SPLITS.items():
    for s in sl:
        assert not (set(s["train_ids"]) & set(s["test_ids"])) and not (set(s["val_ids"]) & set(s["test_ids"]))
        assert not (set(s["train_ids"]) & set(s["val_ids"]))
    print(p, [(s["id"], len(s["train_ids"]), len(s["val_ids"]), len(s["test_ids"])) for s in sl])
assert sorted(c for s in SPLITS["kfold"] for c in s["test_ids"]) == sorted(CASE_IDS), "k-fold must test every case once"
pd.DataFrame([dict(protocol=s["protocol"], split=s["id"], case_id=c, role=r)
              for p in SPLITS for s in SPLITS[p] for r in ("train", "val", "test") for c in s[f"{r}_ids"]]
             ).to_csv(OUTPUT_DIR / "split_assignments.csv", index=False)

# One showcase case per mesh group (median power) -> predictions of every model are saved for figures
SHOWCASE = {}
for g, sub in case_table.groupby("mesh_group"):
    s = sub.sort_values(["power_parameter_value", "mass_flow_kg_s", "case_id"]).reset_index(drop=True)
    SHOWCASE[g] = s.case_id.iloc[len(s) // 2]
print("Showcase cases:", SHOWCASE)


## Shared target transform, split data on GPU, metrics, resumable result store
All learned models share one target transform: asinh for the two heavy-tailed turbulence fields, then standardization. Errors are always computed in original physical units.

In [ ]:
# Target scaler and split data
class TargetScaler:
    def __init__(self, Y):
        self.heavy = np.array([t in HEAVY_TAILED for t in TARGETS])
        s = np.ones(NT); s[self.heavy] = np.maximum(np.median(np.abs(Y[:, self.heavy]), axis=0), 1e-12)
        self.s = s
        Z = self._f(Y.astype(np.float64))
        self.mean = Z.mean(0); self.std = np.maximum(Z.std(0), 1e-12)
        f32 = lambda a: torch.as_tensor(a, dtype=torch.float32, device=DEVICE)
        self.s_t, self.mean_t, self.std_t = f32(self.s), f32(self.mean), f32(self.std)
        self.h_t = torch.as_tensor(self.heavy, device=DEVICE)
    def _f(self, Y):
        Z = Y.copy(); Z[:, self.heavy] = np.arcsinh(Y[:, self.heavy] / self.s[self.heavy]); return Z
    def fwd_np(self, Y):
        return ((self._f(Y.astype(np.float64)) - self.mean) / self.std).astype(np.float32)
    def inv_np(self, Z):
        Y = Z.astype(np.float64) * self.std + self.mean
        Y[:, self.heavy] = self.s[self.heavy] * np.sinh(np.clip(Y[:, self.heavy], -30, 30))
        return Y.astype(np.float32)
    def fwd_t(self, Y):
        Z = Y.clone(); Z[:, self.h_t] = torch.asinh(Y[:, self.h_t] / self.s_t[self.h_t])
        return (Z - self.mean_t) / self.std_t
    def inv_t(self, Z):
        Y = Z * self.std_t + self.mean_t
        Y[:, self.h_t] = self.s_t[self.h_t] * torch.sinh(Y[:, self.h_t].clamp(-30, 30))
        return Y

class SplitData:
    """Scalers + (optionally) all training nodes on GPU. Training samples are drawn case-balanced."""
    def __init__(self, split, pack=False):
        self.split = split
        tr, va = split["train_ids"], split["val_ids"]
        self.pos_tr = np.array([CASE_POS[c] for c in tr])
        self.cf_mean = CASE_FEAT[self.pos_tr].mean(0)
        sc = CASE_FEAT[self.pos_tr].std(0); sc[sc < 1e-8] = 1.0; self.cf_scale = sc
        self.cond_np = ((CASE_FEAT - self.cf_mean) / self.cf_scale).astype(np.float32)
        self.cond = torch.from_numpy(self.cond_np).to(DEVICE)
        rng = np.random.default_rng(SEED)
        samp = [RAW_Y[c][rng.choice(len(RAW_Y[c]), min(len(RAW_Y[c]), TS_SAMPLE), replace=False)] for c in tr]
        self.ts = TargetScaler(np.concatenate(samp))
        self.packed = False
        if pack:
            self.pack()
    def pack(self):
        if self.packed:
            return
        tr, va = self.split["train_ids"], self.split["val_ids"]
        rng = np.random.default_rng(SEED + 7)
        self.xyz = torch.from_numpy(np.concatenate([NORM[c] for c in tr])).to(DEVICE)
        self.y = self.ts.fwd_t(torch.from_numpy(np.concatenate([RAW_Y[c] for c in tr])).to(DEVICE))
        lens = np.array([len(NORM[c]) for c in tr])
        self.lens = torch.as_tensor(lens, device=DEVICE)
        self.starts = torch.as_tensor(np.cumsum(lens) - lens, device=DEVICE)
        self.tr_pos = torch.as_tensor(self.pos_tr, device=DEVICE)
        self.span = torch.as_tensor(np.stack([SPAN[c] for c in tr]), dtype=torch.float32, device=DEVICE)
        self.n_tr = len(tr)
        vx, vc, vy = [], [], []
        for c in va:
            n = len(NORM[c]); sel = np.sort(rng.choice(n, min(n, VAL_NODES_PER_CASE), replace=False))
            vx.append(NORM[c][sel]); vy.append(RAW_Y[c][sel])
            vc.append(np.repeat(self.cond_np[CASE_POS[c]][None], len(sel), axis=0))
        self.val_xyz = torch.from_numpy(np.concatenate(vx)).to(DEVICE)
        self.val_cond = torch.from_numpy(np.concatenate(vc)).to(DEVICE)
        self.val_y = self.ts.fwd_t(torch.from_numpy(np.concatenate(vy)).to(DEVICE))
        self.packed = True
    def sample_batch(self, B):
        ci = torch.randint(0, self.n_tr, (B,), device=DEVICE)
        ln = self.lens[ci]
        idx = self.starts[ci] + torch.minimum((torch.rand(B, device=DEVICE) * ln).long(), ln - 1)
        return self.xyz[idx], self.cond[self.tr_pos[ci]], self.y[idx], self.span[ci]
    def free(self):
        for a in ("xyz", "y", "val_xyz", "val_cond", "val_y"):
            if hasattr(self, a):
                delattr(self, a)
        self.packed = False

_SD_CACHE = {}
def get_sd(split, pack=False):
    key = (split["protocol"], split["id"])
    sd = _SD_CACHE.get(key)
    if sd is None:
        for old in _SD_CACHE.values():
            old.free()
        _SD_CACHE.clear(); gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        sd = _SD_CACHE[key] = SplitData(split)
    if pack:
        sd.pack()
    return sd

In [ ]:
# Metrics and result store
def metric_rows(ytrue, ypred, base):
    rho = ytrue[:, 0]
    lo, hi = np.percentile(rho, [10, 90])
    band = (rho > lo + .25 * (hi - lo)) & (rho < lo + .75 * (hi - lo)) if hi > lo else np.zeros(len(rho), bool)
    rows = []
    for j, t in enumerate(TARGETS):
        truth = ytrue[:, j].astype(np.float64); err = ypred[:, j].astype(np.float64) - truth
        sst = float(((truth - truth.mean()) ** 2).sum()); sd = float(truth.std()); sse = float(err @ err)
        mean_truth = float(truth.mean()); mean_prediction = float(ypred[:, j].mean())
        residual_error = err - err.mean()
        is_pressure = t == "pressure_pa"
        r = dict(base, target=t, n_nodes=len(truth), mae=float(np.abs(err).mean()), rmse=float(np.sqrt(sse / len(err))),
                 median_ae=float(np.median(np.abs(err))), bias=float(err.mean()), std_y=sd,
                 mae_over_std=np.nan if is_pressure else (float(np.abs(err).mean() / sd) if sd > 0 else np.nan),
                 r2=np.nan if is_pressure else (1 - sse / sst if sst > 0 else np.nan),
                 sum_y=float(truth.sum()), sum_y2=float(truth @ truth), sum_sq_err=sse,
                 case_mean_y=mean_truth, case_mean_prediction=mean_prediction,
                 pressure_level_abs_error=abs(mean_prediction - mean_truth) if is_pressure else np.nan,
                 pressure_level_sq_error=(mean_prediction - mean_truth) ** 2 if is_pressure else np.nan,
                 pressure_residual_mae=float(np.abs(residual_error).mean()) if is_pressure else np.nan,
                 pressure_residual_rmse=float(np.sqrt(np.mean(residual_error ** 2))) if is_pressure else np.nan,
                 pressure_residual_nrmse=float(np.sqrt(np.mean(residual_error ** 2)) / max(sd, 1000.0)) if is_pressure else np.nan)
        if band.sum() >= 100:
            e = err[band]
            r.update(mae_transition=float(np.abs(e).mean()), rmse_transition=float(np.sqrt((e * e).mean())),
                     n_transition=int(band.sum()))
        rows.append(r)
    return rows

RESULTS_PATH = OUTPUT_DIR / "all_case_metrics.csv"
STATUS_PATH = OUTPUT_DIR / "run_status.csv"
HISTORY_PATH = OUTPUT_DIR / "training_history.csv"
RESULT_ROWS, STATUS_ROWS, HISTORY_ROWS = [], [], []
if RESULTS_PATH.exists() and STATUS_PATH.exists():
    RESULT_ROWS = pd.read_csv(RESULTS_PATH).to_dict("records")
    STATUS_ROWS = pd.read_csv(STATUS_PATH).to_dict("records")
    if HISTORY_PATH.exists():
        HISTORY_ROWS = pd.read_csv(HISTORY_PATH).to_dict("records")
    print(f"Resuming: {len(RESULT_ROWS):,} metric rows, {len(STATUS_ROWS)} finished tasks")
DONE = {(r["protocol"], r["split"], r["model"]) for r in STATUS_ROWS if r["status"] == "completed"}

def save_progress():
    pd.DataFrame(RESULT_ROWS).to_csv(RESULTS_PATH, index=False)
    pd.DataFrame(STATUS_ROWS).to_csv(STATUS_PATH, index=False)
    pd.DataFrame(HISTORY_ROWS).to_csv(HISTORY_PATH, index=False)

def safe(s):
    return "".join(ch if ch.isalnum() else "_" for ch in s)


## Pooled point-wise models: trees, linear, DeepONet, PINNs, Latent Neural Operator, proposed FiLM-MLP
Every neural model uses the **same** sampler, optimizer, seed, schedule (warm-up + cosine on the larger of step- and wall-clock progress), validation checkpointing, and `SAMPLE_BUDGET`. The two PINNs differ only in the continuity term.

In [ ]:
# Baselines and tree ensembles (pooled over all training cases)
def tree_training_arrays(sd):
    rng = np.random.default_rng(SEED + 3)
    Xs, Ys = [], []
    for c in sd.split["train_ids"]:
        n = len(NORM[c]); sel = rng.choice(n, min(n, TREE_NODES_PER_CASE), replace=False)
        Xs.append(np.concatenate([np.repeat(sd.cond_np[CASE_POS[c]][None], len(sel), 0), NORM[c][sel]], axis=1))
        Ys.append(RAW_Y[c][sel])
    return np.concatenate(Xs).astype(np.float32), np.concatenate(Ys)

import subprocess, sys, tempfile
EXT_SCRIPT = r"""
import sys, json, numpy as np
cfg = json.load(open(sys.argv[1]))
X = np.load(cfg["X"]); Y = np.load(cfg["Y"]); T = np.load(cfg["T"])
gpu = cfg["gpu"]
if cfg["kind"] == "xgb":
    from xgboost import XGBRegressor
    models = []
    for j in range(Y.shape[1]):
        m = XGBRegressor(n_estimators=cfg["trees"], max_depth=8, learning_rate=0.08, subsample=0.85, colsample_bytree=0.85,
                         reg_lambda=2.0, tree_method="hist", device="cuda" if gpu else "cpu", n_jobs=cfg["n_jobs"], random_state=cfg["seed"])
        models.append(m.fit(X, Y[:, j]))
    preds = {k: np.column_stack([m.predict(T[k]) for m in models]) for k in T.files}
else:
    from catboost import CatBoostRegressor
    kw = dict(task_type="GPU", devices="0") if gpu else dict(task_type="CPU", thread_count=cfg["n_jobs"])
    m = CatBoostRegressor(iterations=cfg["trees"], depth=8, learning_rate=0.08, loss_function="MultiRMSE", verbose=False,
                          random_seed=cfg["seed"], allow_writing_files=False, **kw).fit(X, Y)
    preds = {k: m.predict(T[k]) for k in T.files}
np.savez(cfg["out"], **{k: np.asarray(v, dtype=np.float32) for k, v in preds.items()})
"""

def external_boost(kind, X, Y, test_X, attempts=None):
    """Fit XGBoost/CatBoost in a separate process (own CUDA context, own GPU). A CUDA fault there cannot kill this kernel;
    on failure it retries on CPU."""
    work = Path(tempfile.mkdtemp(prefix="ext_", dir=str(OUTPUT_DIR)))
    try:
        np.save(work / "X.npy", X); np.save(work / "Y.npy", Y); np.savez(work / "T.npz", **test_X)
        (work / "run.py").write_text(EXT_SCRIPT)
        attempts = attempts if attempts is not None else ([True, False] if N_GPUS else [False])
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        for use_gpu in attempts:
            cfg = dict(kind=kind, gpu=use_gpu, X=str(work / "X.npy"), Y=str(work / "Y.npy"), T=str(work / "T.npz"),
                       out=str(work / "out.npz"), trees=XGB_TREES if kind == "xgb" else CAT_ITERS, n_jobs=N_CPU, seed=SEED)
            (work / "cfg.json").write_text(json.dumps(cfg))
            env = os.environ.copy()
            env["CUDA_VISIBLE_DEVICES"] = ("1" if N_GPUS > 1 else "0") if use_gpu else ""
            limit = max(120, min(2400 if not use_gpu else 1500, time_remaining() - 300))
            try:
                r = subprocess.run([sys.executable, str(work / "run.py"), str(work / "cfg.json")], env=env,
                                   capture_output=True, text=True, timeout=limit)
                if r.returncode == 0 and (work / "out.npz").exists():
                    out = np.load(work / "out.npz"); return {k: out[k] for k in out.files}, ("GPU" if use_gpu else "CPU")
                print(f"  {kind} {'GPU' if use_gpu else 'CPU'} subprocess failed (code {r.returncode}): {r.stderr.strip()[-300:]}", flush=True)
            except subprocess.TimeoutExpired:
                print(f"  {kind} {'GPU' if use_gpu else 'CPU'} subprocess timed out after {limit:.0f}s", flush=True)
        raise RuntimeError(f"{kind} failed on all devices")
    finally:
        import shutil; shutil.rmtree(work, ignore_errors=True)

def case_feature_matrix(sd, cid):
    return np.concatenate([np.repeat(sd.cond_np[CASE_POS[cid]][None], len(NORM[cid]), 0), NORM[cid]], 1).astype(np.float32)

def make_estimator(name):
    if name == "Ridge":
        return make_pipeline(StandardScaler(), Ridge(alpha=10.0))
    if name == "Random Forest":
        return RandomForestRegressor(n_estimators=RF_TREES, max_depth=20, max_leaf_nodes=TREE_MAX_LEAF, min_samples_leaf=TREE_MIN_LEAF,
                                     max_features=0.8, bootstrap=True, max_samples=TREE_BOOT, n_jobs=N_CPU, random_state=SEED)
    if name == "Extra Trees":
        return ExtraTreesRegressor(n_estimators=ET_TREES, max_depth=20, max_leaf_nodes=TREE_MAX_LEAF, min_samples_leaf=TREE_MIN_LEAF,
                                   max_features=0.8, bootstrap=True, max_samples=TREE_BOOT, n_jobs=N_CPU, random_state=SEED)
    raise KeyError(name)

def run_tabular(name):
    def runner(split):
        sd = get_sd(split)
        t0 = time.monotonic()
        if name == "Mean baseline":
            _, Y = tree_training_arrays(sd); mu = Y.mean(0)
            return (lambda cid: np.repeat(mu[None], len(NORM[cid]), 0).astype(np.float32)), dict(seconds=time.monotonic() - t0)
        X, Y = tree_training_arrays(sd)
        if name in ("XGBoost", "CatBoost"):
            test_X = {f"c{i}": case_feature_matrix(sd, cid) for i, cid in enumerate(split["test_ids"])}
            raw, dev = external_boost("xgb" if name == "XGBoost" else "cat", X, sd.ts.fwd_np(Y), test_X)
            cache = {cid: sd.ts.inv_np(raw[f"c{i}"]) for i, cid in enumerate(split["test_ids"])}
            return (lambda cid: cache.get(cid)), dict(seconds=time.monotonic() - t0, train_rows=len(X), device=dev)
        est = make_estimator(name).fit(X, sd.ts.fwd_np(Y))
        info = dict(seconds=time.monotonic() - t0, train_rows=len(X))
        def predict(cid):
            cond = sd.cond_np[CASE_POS[cid]]; out = []
            for s in range(0, len(NORM[cid]), 100000):
                xyz = NORM[cid][s:s + 100000]
                out.append(est.predict(np.concatenate([np.repeat(cond[None], len(xyz), 0), xyz], 1).astype(np.float32)))
            return sd.ts.inv_np(np.concatenate(out).astype(np.float32))
        return predict, info
    return runner

In [ ]:
# Neural point-wise architectures
def mlp(i, w, d, o):
    layers, k = [], i
    for _ in range(d):
        layers += [nn.Linear(k, w), nn.LayerNorm(w), nn.GELU()]; k = w
    layers.append(nn.Linear(k, o))
    return nn.Sequential(*layers)

NB = 4
def fourier(xyz, bands):
    ph = xyz[:, :, None] * bands[None, None, :]
    return torch.cat([torch.sin(ph), torch.cos(ph)], dim=-1).flatten(1)

class DeepONet(nn.Module):
    def __init__(self):
        super().__init__()
        self.branch = mlp(NF, NN_WIDTH, NN_DEPTH, NN_LATENT * NT); self.trunk = mlp(3, NN_WIDTH, NN_DEPTH, NN_LATENT)
        self.bias = nn.Parameter(torch.zeros(NT))
    def forward(self, cond, xyz):
        b = self.branch(cond).reshape(-1, NT, NN_LATENT)
        return torch.einsum("btl,bl->bt", b, self.trunk(xyz)) / math.sqrt(NN_LATENT) + self.bias

class FourierPINN(nn.Module):
    """Fourier-feature coordinate network; used for both PINN variants (the physics term lives in the loss)."""
    def __init__(self):
        super().__init__()
        self.register_buffer("bands", (2.0 ** torch.arange(NB, dtype=torch.float32)) * math.pi)
        self.net = mlp(NF + 3 * 2 * NB, NN_WIDTH, NN_DEPTH + 1, NT)
    def forward(self, cond, xyz):
        return self.net(torch.cat([cond, fourier(xyz, self.bands)], dim=1))

class LatentNeuralOperator(nn.Module):
    def __init__(self, width=(32 if SMOKE else 128), n_latents=(4 if SMOKE else 24)):
        super().__init__()
        self.L, self.W = n_latents, width
        self.branch = mlp(NF, width, 3, n_latents * width); self.query = mlp(3, width, 3, width)
        self.attn = nn.MultiheadAttention(width, 4, batch_first=True); self.norm = nn.LayerNorm(width)
        self.decoder = mlp(width, width, 2, NT)
    def forward(self, cond, xyz):
        lat = self.branch(cond).reshape(-1, self.L, self.W); q = self.query(xyz).unsqueeze(1)
        att, _ = self.attn(q, lat, lat, need_weights=False)
        return self.decoder(self.norm(q + att)).squeeze(1)

class ConcatNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer("bands", (2.0 ** torch.arange(NB, dtype=torch.float32)) * math.pi)
        self.D, self.W = NN_DEPTH + 1, NN_WIDTH
        self.inp = nn.Linear(3 + 3 * 2 * NB + NF, self.W)
        self.lins = nn.ModuleList([nn.Linear(self.W, self.W) for _ in range(self.D)])
        self.norms = nn.ModuleList([nn.LayerNorm(self.W) for _ in range(self.D)])
        self.out = nn.Linear(self.W, NT)

    def forward(self, cond, xyz):
        features = torch.cat([xyz, fourier(xyz, self.bands), cond], dim=1)
        h = F.gelu(self.inp(features))
        for i in range(self.D):
            h = h + F.gelu(self.norms[i](self.lins[i](h)))
        return self.out(h)


class FiLMNet(nn.Module):
    """Proposed model: coordinate network whose every hidden layer is FiLM-modulated by the case descriptors
    (operating conditions, loop size, orientation flags)."""
    def __init__(self):
        super().__init__()
        self.register_buffer("bands", (2.0 ** torch.arange(NB, dtype=torch.float32)) * math.pi)
        self.D, self.W = NN_DEPTH + 1, NN_WIDTH
        self.inp = nn.Linear(3 + 3 * 2 * NB, self.W)
        self.lins = nn.ModuleList([nn.Linear(self.W, self.W) for _ in range(self.D)])
        self.norms = nn.ModuleList([nn.LayerNorm(self.W) for _ in range(self.D)])
        self.film = mlp(NF, self.W, 2, 2 * self.W * self.D)
        self.out = nn.Linear(self.W, NT)
    def forward(self, cond, xyz):
        gb = self.film(cond).reshape(-1, self.D, 2, self.W)
        h = F.gelu(self.inp(torch.cat([xyz, fourier(xyz, self.bands)], 1)))
        for i in range(self.D):
            g, b = gb[:, i, 0], gb[:, i, 1]
            h = h + F.gelu(self.norms[i](self.lins[i](h)) * (1 + g) + b)
        return self.out(h)


In [ ]:
# Generic trainer (shared by DeepONet, PINNs, LNO, FiLM-MLP)
@torch.no_grad()
def val_loss(model, sd):
    model.eval(); tot, cnt = 0.0, 0
    for s in range(0, len(sd.val_xyz), max(EVAL_CHUNK // MEM_DIV, 256)):
        sl = slice(s, s + max(EVAL_CHUNK // MEM_DIV, 256))
        p = model(sd.val_cond[sl], sd.val_xyz[sl])
        tot += float(((p - sd.val_y[sl]) ** 2).sum()); cnt += p.numel()
    return tot / max(cnt, 1)

PHYS_GEN = torch.Generator(device=DEVICE)      # separate RNG so the physics subset never shifts the data-batch sequence

def continuity_loss(model, sd, xyz, cond, span):
    n = min(PHYS_BATCH, len(xyz)); sel = torch.randperm(len(xyz), device=DEVICE, generator=PHYS_GEN)[:n]
    xp = xyz[sel].clone().requires_grad_(True)
    out = model(cond[sel], xp)
    rho = out[:, 0] * sd.ts.std_t[0] + sd.ts.mean_t[0]
    vel = out[:, 5:8] * sd.ts.std_t[5:8] + sd.ts.mean_t[5:8]
    flux = rho[:, None] * vel
    div = 0.0
    for a in range(3):
        g = torch.autograd.grad(flux[:, a].sum(), xp, create_graph=True)[0][:, a]
        div = div + g / span[sel, a].clamp_min(1e-6)
    scale = (sd.ts.mean_t[0].abs() + sd.ts.std_t[0]) * sd.ts.std_t[5:8].mean() / span.mean()
    return ((div / scale) ** 2).mean()

def lr_at(step, steps, p):
    return LR * min(1.0, (step + 1) / 200) * (0.03 + 0.97 * 0.5 * (1 + math.cos(math.pi * min(p, 1.0))))

def train_pointwise(name, model, sd, split, kind="mse", physics=False, cap=None):
    sd.pack(); PHYS_GEN.manual_seed(SEED)
    model = model.to(DEVICE); B = max(BATCH // MEM_DIV, 64)
    steps = max(SAMPLE_BUDGET // B, 10)
    cap = min(cap or CAP["pointwise"], max(time_remaining() - 120, 5))
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
    best, best_state, step, limited = float("inf"), None, 0, False
    every = max(steps // 12, 1); t0 = time.monotonic()
    while True:
        p = max(step / steps, (time.monotonic() - t0) / cap)
        if p >= 1.0:
            limited = step < steps; break
        for g in opt.param_groups:
            g["lr"] = lr_at(step, steps, p)
        xyz, cond, y, span = sd.sample_batch(B)
        model.train(); opt.zero_grad(set_to_none=True)
        pred = model(cond, xyz)
        loss = F.mse_loss(pred, y) if kind == "mse" else F.huber_loss(pred, y, delta=1.0)
        if physics:
            loss = loss + PHYS_W * continuity_loss(model, sd, xyz, cond, span)
        loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); step += 1
        if step % every == 0 or step == steps:
            v = val_loss(model, sd)
            HISTORY_ROWS.append(dict(protocol=split["protocol"], split=split["id"], model=name, step=step, train_loss=float(loss.detach()), val_loss=v))
            if v < best:
                best, best_state = v, {k: t.detach().clone() for k, t in model.state_dict().items()}
    v = val_loss(model, sd)
    if v < best:
        best, best_state = v, {k: t.detach().clone() for k, t in model.state_dict().items()}
    model.load_state_dict(best_state); model.eval()
    return model, dict(steps=step, planned_steps=steps, budget_limited=limited, best_val=best, seconds=time.monotonic() - t0)

@torch.no_grad()
def predict_pointwise(model, sd, cid):
    xyz = torch.from_numpy(NORM[cid]); cond = sd.cond[CASE_POS[cid]]; out = []
    ch = max(EVAL_CHUNK // MEM_DIV, 256)
    for s in range(0, len(xyz), ch):
        x = xyz[s:s + ch].to(DEVICE); out.append(model(cond.expand(len(x), -1), x))
    return sd.ts.inv_t(torch.cat(out)).cpu().numpy().astype(np.float32)

def run_pointwise(name, ctor, kind="mse", physics=False):
    def runner(split):
        sd = get_sd(split, pack=True)
        seed_all(SEED)                                   # identical init + batch sequence for every neural model of the same type
        model = ctor()
        seed_all(SEED + 1)
        model, info = train_pointwise(name, model, sd, split, kind, physics)
        info["params"] = sum(p.numel() for p in model.parameters())
        return (lambda cid: predict_pointwise(model, sd, cid)), info
    return runner

## Transolver-style, GAMS-GNN-style, F-FNO / U-FNO3D, POD-RBF, nearest-case, lattice floor
* **Transolver-style** is trained exactly as it is evaluated: separate random context and query points from one case.
* **GAMS-GNN-style** is trained and evaluated on random subgraphs of the same size (`GAMS_NODES`).
* **F-FNO / U-FNO3D** work on a lattice of normalized coordinates; the **Lattice floor (oracle)** row shows the error that discretization alone causes.
* **POD-RBF / Nearest case** need aligned meshes, so they are fitted per mesh group (k-fold protocol only).


In [ ]:
# Transolver-style
class TSBlock(nn.Module):
    def __init__(self):
        super().__init__()
        W = TRANS_WIDTH
        self.assign = nn.Linear(W, TRANS_SLICES); self.attn = nn.MultiheadAttention(W, 4, batch_first=True)
        self.n1, self.n2, self.n3 = nn.LayerNorm(W), nn.LayerNorm(W), nn.LayerNorm(W)
        self.ff = nn.Sequential(nn.Linear(W, 2 * W), nn.GELU(), nn.Linear(2 * W, W)); self.local = nn.Sequential(nn.Linear(2 * W, W), nn.GELU())
    def forward(self, hc, hq):
        wc = torch.softmax(self.assign(hc), -1)
        tok = torch.einsum("bnk,bnd->bkd", wc, hc) / (wc.sum(1).unsqueeze(-1) + 1e-6)
        a, _ = self.attn(tok, tok, tok, need_weights=False); tok = self.n1(tok + a); tok = self.n2(tok + self.ff(tok))
        hc = self.n3(hc + self.local(torch.cat([hc, torch.einsum("bnk,bkd->bnd", wc, tok)], -1)))
        wq = torch.softmax(self.assign(hq), -1)
        hq = self.n3(hq + self.local(torch.cat([hq, torch.einsum("bnk,bkd->bnd", wq, tok)], -1)))
        return hc, hq

class TransolverStyle(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = mlp(NF + 3, TRANS_WIDTH, 2, TRANS_WIDTH); self.blocks = nn.ModuleList([TSBlock() for _ in range(TRANS_BLOCKS)])
        self.dec = nn.Linear(TRANS_WIDTH, NT)
    def forward(self, ctx, qry):
        hc, hq = self.enc(ctx), self.enc(qry)
        for b in self.blocks:
            hc, hq = b(hc, hq)
        return self.dec(hq)

def run_transolver(split):
    sd = get_sd(split, pack=True); seed_all(SEED)
    model = TransolverStyle().to(DEVICE); nq = max(TRANS_QRY // MEM_DIV, 32); nc = max(TRANS_CTX // MEM_DIV, 64)
    steps = max(SAMPLE_BUDGET // nq, 10); cap = min(CAP["transolver"], max(time_remaining() - 120, 5))
    opt = torch.optim.AdamW(model.parameters(), lr=LR * 0.6, weight_decay=1e-5)
    rng = np.random.default_rng(SEED)
    tr = split["train_ids"]
    def case_inputs(cid, idx):
        return torch.cat([sd.cond[CASE_POS[cid]].expand(len(idx), -1), torch.from_numpy(NORM[cid][idx]).to(DEVICE)], 1)
    vsets = []
    for cid in split["val_ids"]:
        n = len(NORM[cid]); ci = np.sort(rng.choice(n, min(nc, n), replace=False)); qi = np.sort(rng.choice(n, min(VAL_NODES_PER_CASE, n), replace=False))
        vsets.append((cid, ci, qi, sd.ts.fwd_t(torch.from_numpy(RAW_Y[cid][qi]).to(DEVICE))))
    @torch.no_grad()
    def vloss():
        model.eval(); tot = cnt = 0.0
        for cid, ci, qi, y in vsets:
            p = model(case_inputs(cid, ci)[None], case_inputs(cid, qi)[None])[0]
            tot += float(((p - y) ** 2).sum()); cnt += p.numel()
        return tot / max(cnt, 1)
    best, state, step, limited, t0, every = float("inf"), None, 0, False, time.monotonic(), max(steps // 12, 1)
    while True:
        p = max(step / steps, (time.monotonic() - t0) / cap)
        if p >= 1.0:
            limited = step < steps; break
        for g in opt.param_groups:
            g["lr"] = lr_at(step, steps, p) * 0.6
        cid = tr[rng.integers(len(tr))]; n = len(NORM[cid])
        ci = rng.choice(n, min(nc, n), replace=False); qi = rng.choice(n, min(nq, n), replace=False)
        y = sd.ts.fwd_t(torch.from_numpy(RAW_Y[cid][qi]).to(DEVICE))
        model.train(); opt.zero_grad(set_to_none=True)
        loss = F.mse_loss(model(case_inputs(cid, ci)[None], case_inputs(cid, qi)[None])[0], y)
        loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); step += 1
        if step % every == 0 or step == steps:
            v = vloss(); HISTORY_ROWS.append(dict(protocol=split["protocol"], split=split["id"], model="Transolver-style", step=step, train_loss=float(loss.detach()), val_loss=v))
            if v < best:
                best, state = v, {k: t.detach().clone() for k, t in model.state_dict().items()}
    v = vloss()
    if v < best:
        best, state = v, {k: t.detach().clone() for k, t in model.state_dict().items()}
    model.load_state_dict(state); model.eval()
    @torch.no_grad()
    def predict(cid):
        n = len(NORM[cid]); r = np.random.default_rng(SEED + 5)
        ci = np.sort(r.choice(n, min(nc, n), replace=False)); ctx = case_inputs(cid, ci)[None]; out = []
        ch = max(EVAL_CHUNK // MEM_DIV, 256)
        for s in range(0, n, ch):
            out.append(model(ctx, case_inputs(cid, np.arange(s, min(s + ch, n))) [None])[0])
        return sd.ts.inv_t(torch.cat(out)).cpu().numpy().astype(np.float32)
    return predict, dict(steps=step, planned_steps=steps, budget_limited=limited, best_val=best, seconds=time.monotonic() - t0,
                         params=sum(p.numel() for p in model.parameters()))

In [ ]:
# GAMS-GNN-style (multiscale kNN message passing on random subgraphs)
class EdgeMP(nn.Module):
    def __init__(self, W):
        super().__init__()
        self.edge = nn.Sequential(nn.Linear(2 * W + 4, W), nn.LayerNorm(W), nn.ReLU(), nn.Linear(W, W), nn.ReLU())
        self.node = nn.Sequential(nn.Linear(2 * W, W), nn.LayerNorm(W), nn.ReLU(), nn.Linear(W, W)); self.norm = nn.LayerNorm(W)
    def forward(self, h, ei, ea):
        src, dst = ei
        msg = self.edge(torch.cat([h[src], h[dst], ea], 1))
        tot = h.new_zeros(h.shape).index_add_(0, dst, msg)
        cnt = h.new_zeros((len(h), 1)).index_add_(0, dst, torch.ones((len(dst), 1), device=h.device, dtype=h.dtype))
        return self.norm(h + self.node(torch.cat([h, tot / cnt.clamp_min(1)], 1)))

def graph_for(coords, k=GAMS_K):
    nb = cKDTree(coords).query(coords, k=min(k + 1, len(coords)))[1]
    if nb.ndim == 1:
        nb = nb[:, None]
    dst = np.repeat(np.arange(len(coords)), nb.shape[1]); src = nb.reshape(-1); keep = src != dst
    src, dst = src[keep], dst[keep]; d = coords[src] - coords[dst]
    dist = np.sqrt(np.maximum((d * d).sum(1, keepdims=True), 1e-12))
    return (torch.as_tensor(np.stack([src, dst]), dtype=torch.long, device=DEVICE),
            torch.as_tensor(np.concatenate([d, dist], 1), dtype=torch.float32, device=DEVICE))

def voxel_parent(coords, bins):
    low = coords.min(0); span = np.maximum(coords.max(0) - low, 1e-6)
    keys = np.minimum(np.floor((coords - low) / span * np.asarray(bins)).astype(np.int64), np.asarray(bins) - 1)
    _, parent = np.unique(keys, axis=0, return_inverse=True)
    parent = parent.reshape(-1)
    return parent, int(parent.max() + 1)

def pool_np(vals, parent, n):
    out = np.zeros((n, vals.shape[1]), np.float32); np.add.at(out, parent, vals)
    return out / np.maximum(np.bincount(parent, minlength=n)[:, None], 1)

def make_pyramid(coords):
    e0, a0 = graph_for(coords)
    p1, n1 = voxel_parent(coords, GAMS_BINS1); c1 = pool_np(coords, p1, n1); e1, a1 = graph_for(c1)
    p2, n2 = voxel_parent(c1, GAMS_BINS2); c2 = pool_np(c1, p2, n2); e2, a2 = graph_for(c2)
    t = lambda a: torch.as_tensor(a, dtype=torch.long, device=DEVICE)
    return e0, a0, t(p1), n1, e1, a1, t(p2), n2, e2, a2

def pool_t(h, parent, n):
    out = h.new_zeros((n, h.shape[1])).index_add_(0, parent, h)
    return out / h.new_zeros((n, 1)).index_add_(0, parent, torch.ones((len(parent), 1), device=h.device)).clamp_min(1)

class GAMSStyle(nn.Module):
    def __init__(self):
        super().__init__()
        W = GAMS_WIDTH
        self.enc = mlp(NF + 3, W, 2, W); self.dec = mlp(W, W, 2, NT)
        self.loc = nn.ModuleList([EdgeMP(W) for _ in range(GAMS_BLOCKS)]); self.mid = nn.ModuleList([EdgeMP(W) for _ in range(GAMS_BLOCKS)])
        self.crs = nn.ModuleList([EdgeMP(W) for _ in range(GAMS_BLOCKS)])
        self.fuse = nn.ModuleList([nn.Sequential(nn.Linear(3 * W, W), nn.GELU(), nn.Linear(W, W)) for _ in range(GAMS_BLOCKS)])
        self.norm = nn.ModuleList([nn.LayerNorm(W) for _ in range(GAMS_BLOCKS)])
    def forward(self, x, pyr):
        e0, a0, p1, n1, e1, a1, p2, n2, e2, a2 = pyr
        h = self.enc(x)
        for i in range(GAMS_BLOCKS):
            h = self.loc[i](h, e0, a0)
            h1 = self.mid[i](pool_t(h, p1, n1), e1, a1)
            h2 = self.crs[i](pool_t(h1, p2, n2), e2, a2)
            h = self.norm[i](h + self.fuse[i](torch.cat([h, h1[p1], h2[p2][p1]], 1)))
        return self.dec(h)

def run_gams(split):
    sd = get_sd(split, pack=False); seed_all(SEED)
    model = GAMSStyle().to(DEVICE); rng = np.random.default_rng(SEED + 11); tr = split["train_ids"]
    nn_nodes = max(GAMS_NODES // MEM_DIV, 128)
    steps = max(SAMPLE_BUDGET // nn_nodes, 10); cap = min(CAP["gams"], max(time_remaining() - 120, 5))
    opt = torch.optim.AdamW(model.parameters(), lr=LR * 0.4, weight_decay=1e-5)
    def batch(cid, idx):
        xyz = NORM[cid][idx]
        x = torch.cat([sd.cond[CASE_POS[cid]].expand(len(idx), -1), torch.from_numpy(xyz).to(DEVICE)], 1)
        return x, make_pyramid(xyz)
    vsets = []
    for cid in split["val_ids"]:
        n = len(NORM[cid]); idx = np.sort(rng.choice(n, min(nn_nodes, n), replace=False))
        x, pyr = batch(cid, idx); vsets.append((x, pyr, sd.ts.fwd_t(torch.from_numpy(RAW_Y[cid][idx]).to(DEVICE))))
    @torch.no_grad()
    def vloss():
        model.eval(); return float(np.mean([float(F.mse_loss(model(x, pyr), y)) for x, pyr, y in vsets]))
    best, state, step, limited, t0, every = float("inf"), None, 0, False, time.monotonic(), max(steps // 10, 1)
    while True:
        p = max(step / steps, (time.monotonic() - t0) / cap)
        if p >= 1.0:
            limited = step < steps; break
        for g in opt.param_groups:
            g["lr"] = lr_at(step, steps, p) * 0.4
        cid = tr[rng.integers(len(tr))]; n = len(NORM[cid]); idx = np.sort(rng.choice(n, min(nn_nodes, n), replace=False))
        x, pyr = batch(cid, idx); y = sd.ts.fwd_t(torch.from_numpy(RAW_Y[cid][idx]).to(DEVICE))
        model.train(); opt.zero_grad(set_to_none=True)
        loss = F.mse_loss(model(x, pyr), y); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); step += 1
        if step % every == 0 or step == steps:
            v = vloss(); HISTORY_ROWS.append(dict(protocol=split["protocol"], split=split["id"], model="GAMS-GNN-style", step=step, train_loss=float(loss.detach()), val_loss=v))
            if v < best:
                best, state = v, {k: t.detach().clone() for k, t in model.state_dict().items()}
    v = vloss()
    if v < best:
        best, state = v, {k: t.detach().clone() for k, t in model.state_dict().items()}
    model.load_state_dict(state); model.eval()
    @torch.no_grad()
    def predict(cid):
        n = len(NORM[cid]); parts = np.array_split(np.random.default_rng(SEED + 17).permutation(n), max(1, math.ceil(n / nn_nodes)))
        out = np.zeros((n, NT), np.float32)
        for idx in parts:
            idx = np.sort(idx); x, pyr = batch(cid, idx)
            out[idx] = sd.ts.inv_t(model(x, pyr)).cpu().numpy()
        return out
    return predict, dict(steps=step, planned_steps=steps, budget_limited=limited, best_val=best, seconds=time.monotonic() - t0,
                         params=sum(p.numel() for p in model.parameters()))

In [ ]:
# Lattice (F-FNO / U-FNO3D) on normalized coordinates
LAT_ARR = np.array(LAT)
LAT_FLAT, LAT_COUNT = {}, {}
for _cid in CASE_IDS:
    ix = np.minimum((NORM[_cid] * LAT_ARR).astype(np.int64), LAT_ARR - 1)
    fl = (ix[:, 0] * LAT[1] + ix[:, 1]) * LAT[2] + ix[:, 2]
    LAT_FLAT[_cid] = fl.astype(np.int64); LAT_COUNT[_cid] = np.bincount(fl, minlength=int(np.prod(LAT))).astype(np.float32)
_cc = [np.linspace(0.5 / d, 1 - 0.5 / d, d, dtype=np.float32) for d in LAT]
LAT_COORD = torch.from_numpy(np.stack(np.meshgrid(*_cc, indexing="ij"))).to(DEVICE)       # (3, Dx, Dy, Dz)

class SpectralAxis(nn.Module):
    def __init__(self, width, axis, size):
        super().__init__()
        self.axis, self.m = axis, min(FNO_MODES, size // 2 + 1)
        self.w = nn.Parameter(torch.randn(width, width, self.m, 2) * 0.02)
    def forward(self, x):
        xf = torch.fft.rfft(x, dim=self.axis, norm="ortho").movedim(self.axis, -1)
        m = min(self.m, xf.shape[-1]); w = torch.view_as_complex(self.w[:, :, :m].contiguous())
        out = torch.zeros_like(xf); out[..., :m] = torch.einsum("bi...m,iom->bo...m", xf[..., :m], w)
        return torch.fft.irfft(out.movedim(-1, self.axis), n=x.shape[self.axis], dim=self.axis, norm="ortho")

class FBlock(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.spec = nn.ModuleList([SpectralAxis(width, 2 + a, LAT[a]) for a in range(3)])
        self.local = nn.Conv3d(width, width, 1); self.norm = nn.GroupNorm(min(4, width), width)
    def forward(self, x):
        y = self.local(x)
        for s in self.spec:
            y = y + s(x)
        return x + F.gelu(self.norm(y))

class FFNO3D(nn.Module):
    def __init__(self, cin, width=FNO_WIDTH, depth=4):
        super().__init__()
        self.lift = nn.Conv3d(cin, width, 1); self.blocks = nn.ModuleList([FBlock(width) for _ in range(depth)])
        self.proj = nn.Sequential(nn.Conv3d(width, width, 1), nn.GELU(), nn.Conv3d(width, NT, 1))
    def forward(self, x):
        h = self.lift(x)
        for b in self.blocks:
            h = b(h)
        return self.proj(h)

class UFNO3D(nn.Module):
    def __init__(self, cin, width=FNO_WIDTH):
        super().__init__()
        self.lift = nn.Conv3d(cin, width, 1); self.blocks = nn.ModuleList([FBlock(width) for _ in range(3)])
        self.down = nn.Sequential(nn.Conv3d(width, 2 * width, 3, stride=2, padding=1), nn.GELU(), nn.Conv3d(2 * width, 2 * width, 3, padding=1), nn.GELU())
        self.up = nn.Sequential(nn.Conv3d(2 * width, width, 1), nn.GELU()); self.fix = nn.Conv3d(width, width, 1)
        self.proj = nn.Sequential(nn.Conv3d(width, width, 1), nn.GELU(), nn.Conv3d(width, NT, 1))
    def forward(self, x):
        h = self.lift(x)
        for b in self.blocks:
            h = b(h)
        u = F.interpolate(self.up(self.down(h)), size=h.shape[-3:], mode="trilinear", align_corners=False)
        return self.proj(h + self.fix(h + u))

def lattice_grid(cid, sd, with_target=True):
    fl = torch.from_numpy(LAT_FLAT[cid]).to(DEVICE); cnt = torch.from_numpy(LAT_COUNT[cid]).to(DEVICE)
    cells = int(np.prod(LAT)); y = None
    if with_target:
        z = sd.ts.fwd_t(torch.from_numpy(RAW_Y[cid]).to(DEVICE))
        y = (torch.zeros(cells, NT, device=DEVICE).index_add_(0, fl, z) / cnt.clamp_min(1)[:, None]).T.reshape(NT, *LAT)
    return fl, cnt.reshape(1, *LAT), y

def lattice_input(cid, sd, cnt):
    cond = sd.cond[CASE_POS[cid]][:, None, None, None].expand(-1, *LAT)
    return torch.cat([LAT_COORD, cond, (cnt > 0).float()], 0)

def run_fno(name, ctor):
    def runner(split):
        sd = get_sd(split); seed_all(SEED)
        model = ctor(3 + NF + 1).to(DEVICE); rng = np.random.default_rng(SEED)
        cap = min(CAP["fno"], max(time_remaining() - 120, 5)); steps = FNO_STEPS
        opt = torch.optim.AdamW(model.parameters(), lr=LR * 0.4, weight_decay=1e-5)
        cache = {}
        def get(cid):
            if cid not in cache:
                fl, cnt, y = lattice_grid(cid, sd); cache[cid] = (lattice_input(cid, sd, cnt), y, cnt)
            return cache[cid]
        def wmse(pred, y, cnt):
            return (((pred - y) ** 2) * cnt).sum() / (cnt.sum() * NT).clamp_min(1)
        @torch.no_grad()
        def vloss():
            model.eval(); return float(np.mean([float(wmse(model(get(c)[0][None])[0], get(c)[1], get(c)[2])) for c in split["val_ids"]]))
        best, state, step, limited, t0, every = float("inf"), None, 0, False, time.monotonic(), max(steps // 8, 1)
        while True:
            p = max(step / steps, (time.monotonic() - t0) / cap)
            if p >= 1.0:
                limited = step < steps; break
            for g in opt.param_groups:
                g["lr"] = lr_at(step, steps, p) * 0.4
            ids = [split["train_ids"][i] for i in rng.integers(len(split["train_ids"]), size=FNO_BATCH)]
            xb = torch.stack([get(c)[0] for c in ids]); yb = torch.stack([get(c)[1] for c in ids]); cb = torch.stack([get(c)[2] for c in ids])
            model.train(); opt.zero_grad(set_to_none=True)
            loss = wmse(model(xb), yb, cb); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); step += 1
            if step % every == 0 or step == steps:
                v = vloss(); HISTORY_ROWS.append(dict(protocol=split["protocol"], split=split["id"], model=name, step=step, train_loss=float(loss.detach()), val_loss=v))
                if v < best:
                    best, state = v, {k: t.detach().clone() for k, t in model.state_dict().items()}
        v = vloss()
        if v < best:
            best, state = v, {k: t.detach().clone() for k, t in model.state_dict().items()}
        model.load_state_dict(state); model.eval()
        @torch.no_grad()
        def predict(cid):
            fl, cnt, _ = lattice_grid(cid, sd, with_target=False)
            grid = model(lattice_input(cid, sd, cnt)[None])[0].reshape(NT, -1)
            return sd.ts.inv_t(grid[:, fl].T).cpu().numpy().astype(np.float32)
        return predict, dict(steps=step, planned_steps=steps, budget_limited=limited, best_val=best, seconds=time.monotonic() - t0,
                             params=sum(p.numel() for p in model.parameters()))
    return runner

def run_lattice_floor(split):
    sd = get_sd(split)
    @torch.no_grad()
    def predict(cid):
        fl, cnt, y = lattice_grid(cid, sd)
        return sd.ts.inv_t(y.reshape(NT, -1)[:, fl].T).cpu().numpy().astype(np.float32)
    return predict, dict(seconds=0.0)

In [ ]:
# POD-RBF and nearest-case (aligned meshes -> fitted per mesh group)
def sqdist(a, b):
    return np.maximum((a * a).sum(1, keepdims=True) + (b * b).sum(1)[None, :] - 2 * a @ b.T, 0.)

def tune_rbf(x, coef):
    n = len(x)
    if n < 3:
        return 1., .1
    d = sqdist(x, x); scale = max(float(np.std(coef)), 1e-12); best = (np.inf, 1., .1); eye = np.eye(n)
    for ell in RBF_LENGTH_SCALES:
        k = np.exp(-.5 * d / (ell * ell))
        for alpha in RBF_ALPHAS:
            inv = np.linalg.solve(k + alpha * eye, eye); res = (inv @ coef) / np.diag(inv)
            sc = float(np.mean((res / scale) ** 2))
            if sc < best[0]:
                best = (sc, ell, alpha)
    return best[1], best[2]

def pod_rbf_predict(x_train, fields_train, x_query):
    n, nodes, nt = fields_train.shape
    pred = np.empty((len(x_query), nodes, nt), np.float32)
    dtr, dq = sqdist(x_train, x_train), sqdist(x_query, x_train)
    for t in range(nt):
        mean = fields_train[:, :, t].mean(0, dtype=np.float64); cen = fields_train[:, :, t].astype(np.float64) - mean
        ev, vec = np.linalg.eigh(cen @ cen.T); o = np.argsort(ev)[::-1]; ev = np.maximum(ev[o], 0.); vec = vec[:, o]
        tot = float(ev.sum())
        if tot <= 1e-20 or n < 2:
            pred[:, :, t] = mean; continue
        rank = max(1, min(n - 1, POD_MAX_MODES, int(np.searchsorted(np.cumsum(ev) / tot, POD_ENERGY) + 1), int(np.sum(ev > max(tot * 1e-12, 1e-20)))))
        basis = (cen.T @ vec[:, :rank]) / np.sqrt(np.maximum(ev[:rank], 1e-20))[None, :]
        coef = cen @ basis; qc = np.zeros((len(x_query), rank))
        for m in range(rank):
            ell, alpha = tune_rbf(x_train, coef[:, m])
            k = np.exp(-.5 * dtr / (ell * ell)) + alpha * np.eye(n)
            qc[:, m] = np.exp(-.5 * dq / (ell * ell)) @ np.linalg.solve(k, coef[:, m])
        pred[:, :, t] = (mean[None, :] + qc @ basis.T).astype(np.float32)
    return pred

def pod_gpr_predict(x_train, fields_train, x_query):
    n, nodes, nt = fields_train.shape
    pred = np.empty((len(x_query), nodes, nt), np.float32)
    dtr, dq = sqdist(x_train, x_train), sqdist(x_query, x_train)
    for t in range(nt):
        mean = fields_train[:, :, t].mean(0, dtype=np.float64)
        centered = fields_train[:, :, t].astype(np.float64) - mean
        eigenvalues, eigenvectors = np.linalg.eigh(centered @ centered.T)
        order = np.argsort(eigenvalues)[::-1]
        eigenvalues = np.maximum(eigenvalues[order], 0.0)
        eigenvectors = eigenvectors[:, order]
        total = float(eigenvalues.sum())
        if total <= 1e-20 or n < 2:
            pred[:, :, t] = mean
            continue
        rank = max(1, min(n - 1, POD_MAX_MODES, int(np.searchsorted(
            np.cumsum(eigenvalues) / total, POD_ENERGY
        ) + 1)))
        basis = (centered.T @ eigenvectors[:, :rank]) / np.sqrt(
            np.maximum(eigenvalues[:rank], 1e-20)
        )[None, :]
        scores = centered @ basis
        kernel_train = np.exp(-0.5 * dtr) + 1e-4 * np.eye(n)
        kernel_query = np.exp(-0.5 * dq)
        coefficients = np.linalg.solve(kernel_train, scores)
        pred[:, :, t] = (mean[None, :] + (kernel_query @ coefficients) @ basis.T).astype(
            np.float32
        )
    return pred

def run_group_method(kind):
    def runner(split):
        t0 = time.monotonic(); preds = {}
        for g in MESH_GROUPS:
            te = [c for c in split["test_ids"] if GROUP_OF[c] == g]; tr = [c for c in split["train_ids"] if GROUP_OF[c] == g]
            if not te or len(tr) < 2 or not GROUP_ALIGNED[g]:
                continue
            ft = CASE_FEAT[[CASE_POS[c] for c in tr]]; mu = ft.mean(0); sc = ft.std(0); sc[sc < 1e-8] = 1.
            xtr = (ft - mu) / sc; xte = (CASE_FEAT[[CASE_POS[c] for c in te]] - mu) / sc
            fields = np.stack([RAW_Y[c] for c in tr])
            if kind == "pod":
                out = pod_rbf_predict(xtr.astype(np.float64), fields, xte.astype(np.float64))
            elif kind == "pod_gpr":
                out = pod_gpr_predict(xtr.astype(np.float64), fields, xte.astype(np.float64))
            else:
                out = fields[sqdist(xte, xtr).argmin(1)]
            for i, c in enumerate(te):
                preds[c] = out[i]
        return (lambda cid: preds.get(cid)), dict(seconds=time.monotonic() - t0, cases_covered=len(preds))
    return runner


## Model registry, schedule, and resumable runner
Schedule: (1) cheap/core models in k-fold, (2) leave-one-mesh-group-out for the pooled subset, (3) heavier models in k-fold. Each (model, split) is checkpointed; rerun the cell after a session restart to resume. Expected total on Kaggle T4 ×2: roughly 6–8 h; the hard stop is `MAX_TOTAL_HOURS`.

In [ ]:
# Registry
MODEL_SPECS = [
    # name, runner, stage, runs on geometry holdout?, reference row?, mesh-aligned (per group)?
    dict(name="Mean baseline", fn=run_tabular("Mean baseline"), stage="core", geom=True),
    dict(name="Ridge", fn=run_tabular("Ridge"), stage="core", geom=True),
    dict(name="Lattice floor (oracle)", fn=run_lattice_floor, stage="core", geom=False, ref=True),
    dict(name="XGBoost", fn=run_tabular("XGBoost"), stage="core", geom=True),
    dict(name="DeepONet", fn=run_pointwise("DeepONet", DeepONet), stage="core", geom=True),
    dict(name="PINN data-only", fn=run_pointwise("PINN data-only", FourierPINN), stage="core", geom=True),
    dict(name="PINN continuity", fn=run_pointwise("PINN continuity", FourierPINN, physics=True), stage="core", geom=True),
    dict(name="Domain-informed FiLM-MLP", fn=run_pointwise("Domain-informed FiLM-MLP", FiLMNet, kind="huber"), stage="core", geom=True),
    dict(name="Domain-informed Concat-MLP", fn=run_pointwise("Domain-informed Concat-MLP", ConcatNet, kind="huber"), stage="core", geom=True),
    dict(name="POD-RBF", fn=run_group_method("pod"), stage="core", geom=False, aligned=True),
    dict(name="POD-GPR", fn=run_group_method("pod_gpr"), stage="core", geom=False, aligned=True),
    dict(name="Nearest case", fn=run_group_method("nearest"), stage="core", geom=False, aligned=True),
    dict(name="Random Forest", fn=run_tabular("Random Forest"), stage="heavy", geom=True),
    dict(name="Extra Trees", fn=run_tabular("Extra Trees"), stage="heavy", geom=True),
    dict(name="CatBoost", fn=run_tabular("CatBoost"), stage="heavy", geom=True),
    dict(name="Latent Neural Operator", fn=run_pointwise("Latent Neural Operator", LatentNeuralOperator), stage="heavy", geom=True),
    dict(name="Transolver-style", fn=run_transolver, stage="heavy", geom=True),
    dict(name="F-FNO", fn=run_fno("F-FNO", FFNO3D), stage="heavy", geom=True),
    dict(name="U-FNO3D", fn=run_fno("U-FNO3D", UFNO3D), stage="heavy", geom=True),
    dict(name="GAMS-GNN-style", fn=run_gams, stage="heavy", geom=True),
]
REFERENCE_MODELS = {s["name"] for s in MODEL_SPECS if s.get("ref")}
ALIGNED_MODELS = {s["name"] for s in MODEL_SPECS if s.get("aligned")}
TASKS = ([("kfold", s) for s in MODEL_SPECS if s["stage"] == "core"]
         + [("geometry_holdout", s) for s in MODEL_SPECS if s["geom"]]
         + [("kfold", s) for s in MODEL_SPECS if s["stage"] == "heavy"])


In [ ]:
# Runner
def is_oom(exc):
    return isinstance(exc, torch.cuda.OutOfMemoryError) or "out of memory" in str(exc).lower()

def run_one(spec, protocol, split):
    global MEM_DIV
    key = (protocol, split["id"], spec["name"])
    if key in DONE:
        return
    ensure_time(f"{spec['name']} / {split['id']}", need=300)
    status = dict(protocol=protocol, split=split["id"], model=spec["name"], status="failed", note="")
    t0 = time.monotonic()
    for attempt in (1, 2):
        try:
            predict, info = spec["fn"](split)
            rows, n_scored = [], 0
            for cid in split["test_ids"]:
                pred = predict(cid)
                if pred is None:
                    continue
                if not np.isfinite(pred).all():
                    status["note"] += f" non-finite predictions for {cid};"
                    pred = np.nan_to_num(pred, nan=0.0, posinf=0.0, neginf=0.0)
                rows += metric_rows(RAW_Y[cid], pred, dict(protocol=protocol, split=split["id"], model=spec["name"],
                                                           case_id=cid, mesh_group=GROUP_OF[cid]))
                n_scored += 1
                if protocol == "kfold":
                    case_data_path = PRED_DATA_DIR / f"{safe(cid)}.npz"
                    if not case_data_path.exists():
                        np.savez_compressed(case_data_path, coordinates=RAW_C[cid], truth=RAW_Y[cid])
                    if spec["name"] in PRED_MODELS:
                        np.savez_compressed(
                            PRED_DIR / f"{safe(spec['name'])}__{safe(cid)}.npz",
                            prediction=pred.astype(np.float32),
                        )
                if protocol == "kfold" and cid == SHOWCASE.get(GROUP_OF[cid]):
                    np.save(PRED_DIR / f"{safe(spec['name'])}__{cid}.npy", pred.astype(np.float32))
            RESULT_ROWS.extend(rows)
            status.update(status="completed", cases_scored=n_scored, cases_expected=len(split["test_ids"]),
                          seconds=round(time.monotonic() - t0, 1), **{k: v for k, v in info.items() if k != "seconds"})
            break
        except TimeoutError as exc:
            status.update(status="timeout", note=str(exc)); STATUS_ROWS.append(status); save_progress(); raise
        except Exception as exc:
            if is_oom(exc) and attempt == 1:
                MEM_DIV *= 2
                gc.collect(); torch.cuda.empty_cache()
                print(f"  OOM in {spec['name']}: retrying once with batch/chunk sizes / {MEM_DIV}", flush=True)
                continue
            status["note"] = f"{type(exc).__name__}: {str(exc)[:200]}"
            print(f"  FAILED {spec['name']} / {split['id']}: {status['note']}", flush=True)
            break
    STATUS_ROWS.append(status)
    if status["status"] == "completed":
        DONE.add(key)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    save_progress()
    lim = " [budget-limited]" if status.get("budget_limited") else ""
    print(f"[{(time.monotonic()-WALL_START)/3600:5.2f} h] {protocol:>16} {split['id']:<28} {spec['name']:<26} {status['status']}{lim}"
          f" ({status.get('seconds', 0):.0f}s)", flush=True)

try:
    for protocol, spec in TASKS:
        if protocol == "geometry_holdout" and not SPLITS["geometry_holdout"]:
            continue
        for split in SPLITS[protocol]:
            run_one(spec, protocol, split)
        MEM_DIV = max(1, MEM_DIV // 2) if MEM_DIV > 1 else 1      # relax after a successful task
    print("All scheduled tasks finished.")
except TimeoutError as exc:
    print(f"\nStopped by the time guard: {exc}. Partial results are saved; see the completeness audit below.")
save_progress()
print(f"Elapsed: {(time.monotonic()-WALL_START)/3600:.2f} h")


## Completeness audit, comparison tables, uncertainty, geometry holdout, figures
Ranking uses only **non-reference** models that scored **every** case of the protocol. `mae_over_std` is the scale-free error. `gap_ci` is a paired bootstrap over cases (best vs. runner-up); if it includes 0 the two are statistically indistinguishable on these cases.

In [ ]:
# Completeness and summary
res = pd.DataFrame(RESULT_ROWS)
status = pd.DataFrame(STATUS_ROWS)
if res.empty:
    raise RuntimeError("No results available. Check run_status.csv.")
print("Task status:"); display(status.groupby(["protocol", "model", "status"]).size().unstack(fill_value=0))
lim = status[status.get("budget_limited", pd.Series(False, index=status.index)).fillna(False).astype(bool)] if "budget_limited" in status else status.iloc[0:0]
print("Budget-limited tasks (stopped by wall-clock cap before the sample budget):")
display(lim[["protocol", "split", "model", "steps", "planned_steps", "seconds"]] if len(lim) else "none")

def complete_models(protocol):
    d = res[res.protocol == protocol]
    expected = set(c for s in SPLITS[protocol] for c in s["test_ids"])
    n = d.groupby("model").case_id.nunique()
    return sorted(m for m, k in n.items() if k == len(expected) and m not in REFERENCE_MODELS), len(expected), n

def summarize(protocol, models):
    d = res[(res.protocol == protocol) & res.model.isin(models)]
    rows = []
    for (m, t), g in d.groupby(["model", "target"], sort=False):
        n = g.n_nodes.sum(); sse = g.sum_sq_err.sum(); sy = g.sum_y.sum(); sy2 = g.sum_y2.sum(); sst = sy2 - sy * sy / n
        row = dict(model=m, target=t, n_cases=g.case_id.nunique(), mae=g.mae.mean(), rmse=g.rmse.mean(),
                   mae_over_std=g.mae_over_std.mean(), r2_case_mean=g.r2.mean(), pooled_rmse=math.sqrt(sse / n),
                   pooled_r2=1 - sse / sst if sst > 0 else np.nan, mae_transition=g.get("mae_transition", pd.Series(dtype=float)).mean())
        for col in ("mae", "rmse", "mae_over_std"):
            rng = np.random.default_rng(SEED)
            v = g[col].to_numpy(float); v = v[np.isfinite(v)]
            b = rng.choice(v, size=(BOOTSTRAP, len(v)), replace=True).mean(1) if len(v) else np.array([np.nan])
            row[f"{col}_lo"], row[f"{col}_hi"] = np.quantile(b, [0.025, 0.975])
        rows.append(row)
    return pd.DataFrame(rows)

kf_models, kf_expected, kf_counts = complete_models("kfold")
print(f"k-fold: {kf_expected} cases expected. Models with all cases: {kf_models}")
print("Cases scored per model:"); display(kf_counts.sort_values())
summary = summarize("kfold", kf_models) if kf_models else pd.DataFrame()
if len(summary):
    summary.to_csv(OUTPUT_DIR / "kfold_model_comparison.csv", index=False)
    display(summary.sort_values(["target", "mae"]).round(6))

In [ ]:
# Best by target with paired-bootstrap gap to the runner-up
def best_table(summary, protocol, metric="mae"):
    out = []
    for t in TARGETS:
        s = summary[summary.target == t].sort_values(metric)
        if len(s) < 2:
            continue
        b, r = s.iloc[0].model, s.iloc[1].model
        d = res[(res.protocol == protocol) & (res.target == t)].pivot_table(index="case_id", columns="model", values=metric)
        diff = (d[r] - d[b]).dropna().to_numpy()
        bs = np.random.default_rng(SEED).choice(diff, size=(BOOTSTRAP, len(diff)), replace=True).mean(1)
        lo, hi = np.quantile(bs, [0.025, 0.975])
        out.append(dict(target=t, best=b, best_value=s.iloc[0][metric], runner_up=r, runner_value=s.iloc[1][metric],
                        gap_mean=diff.mean(), gap_ci_low=lo, gap_ci_high=hi, distinguishable=bool(lo > 0)))
    return pd.DataFrame(out)

if len(summary):
    best_mae = best_table(summary, "kfold", "mae"); best_rmse = best_table(summary, "kfold", "rmse")
    best_scaled = best_table(summary, "kfold", "mae_over_std")
    print("Best by MAE (distinguishable = paired 95% CI of the gap excludes 0):"); display(best_mae.round(6))
    print("Best by RMSE:"); display(best_rmse.round(6)); print("Best by MAE / std (scale-free):"); display(best_scaled.round(4))
    best_mae.to_csv(OUTPUT_DIR / "best_by_target_mae.csv", index=False); best_rmse.to_csv(OUTPUT_DIR / "best_by_target_rmse.csv", index=False)

In [ ]:
# Reference rows (aligned-mesh methods are not pooled; lattice floor is a reference, not a competitor)
ref_models = [m for m in res.model.unique() if m in REFERENCE_MODELS or m in ALIGNED_MODELS]
if ref_models:
    print("Reference / mesh-aligned rows (cases scored differ; compare with care):")
    rs = summarize("kfold", ref_models)
    display(rs.pivot_table(index="model", columns="target", values="mae_over_std").round(4))

In [ ]:
# Per-mesh-group breakdown and transition-band error
if len(summary):
    grp = res[(res.protocol == "kfold") & res.model.isin(kf_models)].groupby(["model", "mesh_group", "target"]).mae_over_std.mean().unstack("target")
    grp.to_csv(OUTPUT_DIR / "kfold_by_mesh_group_mae_over_std.csv"); print("MAE/std by mesh group:"); display(grp.round(4))
    if "mae_transition" in res:
        tr = res[(res.protocol == "kfold") & res.model.isin(kf_models)].groupby(["model", "target"])[["mae", "mae_transition"]].mean()
        tr["transition_over_all"] = tr.mae_transition / tr.mae
        tr.to_csv(OUTPUT_DIR / "kfold_transition_band_mae.csv"); print("Error in the pseudocritical-transition band vs. whole domain:"); display(tr.round(6))

In [ ]:
# Geometry holdout: can a model transfer to an unseen configuration/geometry?
gm = res[res.protocol == "geometry_holdout"]
if len(gm):
    base = gm[gm.model == "Mean baseline"].groupby(["split", "target"]).mae.mean().rename("mean_baseline_mae")
    g = gm.groupby(["split", "model", "target"]).mae.mean().reset_index().merge(base.reset_index(), on=["split", "target"])
    g["mae_relative_to_mean_baseline"] = g.mae / g.mean_baseline_mae
    gt = g.pivot_table(index=["split", "model"], columns="target", values="mae_relative_to_mean_baseline")
    gt.to_csv(OUTPUT_DIR / "geometry_holdout_relative_mae.csv")
    print("Leave-one-mesh-group-out: MAE relative to the training-mean baseline (<1 = better than predicting the mean; >1 = worse):")
    display(gt.round(3))
else:
    print("Geometry-holdout results not available (not reached or time guard).")

In [ ]:
# Training curves and parameter counts
hist = pd.DataFrame(HISTORY_ROWS)
if len(hist):
    fig, ax = plt.subplots(figsize=(8, 5))
    for m, gdf in hist.groupby("model"):
        a = gdf.groupby("step").val_loss.mean(); ax.plot(a.index, a.values, label=m)
    ax.set_yscale("log"); ax.set_xlabel("step"); ax.set_ylabel("validation MSE (standardized)"); ax.legend(fontsize=7); ax.set_title("Validation curves (mean over splits)")
    fig.savefig(OUTPUT_DIR / "validation_curves.png", dpi=150, bbox_inches="tight"); plt.show()
if "params" in status:
    display(status.dropna(subset=["params"]).groupby("model").params.first().rename("trainable_parameters").to_frame())

## Visual reconstruction of loops by the best model for each parameter
For every target the model with the lowest k-fold MAE (among models that scored all cases) is used. Each figure shows one held-out loop: CFD, the best model's reconstruction, and the absolute error, on the mid-plane slab of the loop. Predictions were saved during the run for one showcase case per mesh group, and each of those cases was held out in exactly one fold.

In [ ]:
# Final cell: loop reconstructions
def loop_slice_idx(cid, max_pts=60000):
    xyz = RAW_C[cid]; n = len(xyz); zspan = max(float(np.ptp(xyz[:, 2])), 1e-9); zmed = float(np.median(xyz[:, 2]))
    for frac in (0.02, 0.05, 0.1, 0.2, 0.5, 1.0):
        idx = np.flatnonzero(np.abs(xyz[:, 2] - zmed) <= frac * zspan / 2)
        if len(idx) >= min(3000, max(int(0.05 * n), 1)):
            break
    if len(idx) > max_pts:
        idx = np.sort(np.random.default_rng(SEED).choice(idx, max_pts, replace=False))
    return idx

def best_models_by_mae():
    if "best_mae" in globals() and len(best_mae):
        return dict(zip(best_mae.target, best_mae.best))
    if "summary" in globals() and len(summary):
        return {t: summary[summary.target == t].sort_values("mae").iloc[0].model for t in TARGETS if (summary.target == t).any()}
    return {}

def reconstruction_figure(cid, best_models):
    idx = loop_slice_idx(cid); xyz = RAW_C[cid][idx]
    fig, axs = plt.subplots(4, 6, figsize=(27, 18), constrained_layout=True)
    for k, t in enumerate(TARGETS):
        r, c0 = k // 2, (k % 2) * 3; m = best_models.get(t)
        f = PRED_DIR / f"{safe(m)}__{cid}.npy" if m else None
        act = RAW_Y[cid][idx, k]
        if f is None or not f.exists():
            for c in range(3):
                axs[r, c0 + c].axis("off")
            axs[r, c0].text(0.0, 0.5, f"{t}: no saved prediction for {m}", transform=axs[r, c0].transAxes)
            continue
        pred = np.load(f)[idx, k]; err = np.abs(pred - act)
        lo, hi = np.nanpercentile(np.r_[act, pred], [1, 99]); hi = hi if hi > lo else lo + 1e-12
        panels = ((act, f"CFD: {t}", (lo, hi), "inferno"), (pred, f"{m}", (lo, hi), "inferno"),
                  (err, f"|error|  (MAE {err.mean():.3g})", (0, max(np.nanpercentile(err, 99), 1e-12)), "viridis"))
        for c, (v, ttl, (a, b), cm) in enumerate(panels):
            ax = axs[r, c0 + c]; s = ax.scatter(xyz[:, 0], xyz[:, 1], c=v, s=5, cmap=cm, vmin=a, vmax=b, linewidths=0)
            ax.set_aspect("equal"); ax.set_title(ttl, fontsize=10); ax.tick_params(labelsize=7); fig.colorbar(s, ax=ax, shrink=0.8)
    fig.suptitle(f"Held-out loop {cid}  |  {GROUP_OF[cid]}  |  best model per parameter (k-fold MAE)", fontsize=15)
    out = OUTPUT_DIR / f"reconstruction_{safe(cid)}.png"; fig.savefig(out, dpi=140, bbox_inches="tight"); plt.show(); plt.close(fig)

_bm = best_models_by_mae()
if not _bm:
    print("No ranked models yet, so no reconstruction can be drawn (see the completeness audit above).")
else:
    print("Best model per parameter:", _bm)
    for _g, _cid in SHOWCASE.items():
        reconstruction_figure(_cid, _bm)
print("Outputs written to", OUTPUT_DIR)

## Proposed Loop-Unrolled Offset-Shape Operator (pending geometry input)

The prototype below defines the gated architecture and differentiable EOS lookup. Training and score ingestion remain pending a PhD-scholar centerline/path export containing `s_star`, `r_star`, `theta`, leg type, heater/cooler indicators, and velocity rotation. Do not treat the prototype definition as an experiment or activate the proposed-method claim until the prespecified paired-bootstrap gate passes.

The four required ablations are no loop unrolling, no level/shape split, no EOS-derived density, and no periodic encoding. Their result cells remain pending because the model cannot be trained or scored until the geometry metadata is validated.


In [ ]:
class DifferentiableCO2EOS(nn.Module):
    def __init__(self, temperature_k, pressure_pa, density_kg_m3):
        super().__init__()
        self.register_buffer("temperature_k", torch.as_tensor(temperature_k, dtype=torch.float32))
        self.register_buffer("pressure_pa", torch.as_tensor(pressure_pa, dtype=torch.float32))
        self.register_buffer("density_kg_m3", torch.as_tensor(density_kg_m3, dtype=torch.float32))

    def forward(self, temperature_k, pressure_pa):
        t = temperature_k.clamp(self.temperature_k[0], self.temperature_k[-1])
        p = pressure_pa.clamp(self.pressure_pa[0], self.pressure_pa[-1])
        ti = torch.bucketize(t.contiguous(), self.temperature_k).clamp(1, len(self.temperature_k) - 1)
        pi = torch.bucketize(p.contiguous(), self.pressure_pa).clamp(1, len(self.pressure_pa) - 1)
        t0, t1 = self.temperature_k[ti - 1], self.temperature_k[ti]
        p0, p1 = self.pressure_pa[pi - 1], self.pressure_pa[pi]
        wt = ((t - t0) / (t1 - t0).clamp_min(1e-8)).clamp(0, 1)
        wp = ((p - p0) / (p1 - p0).clamp_min(1e-8)).clamp(0, 1)
        d00 = self.density_kg_m3[ti - 1, pi - 1]
        d01 = self.density_kg_m3[ti - 1, pi]
        d10 = self.density_kg_m3[ti, pi - 1]
        d11 = self.density_kg_m3[ti, pi]
        return (1 - wt) * ((1 - wp) * d00 + wp * d01) + wt * ((1 - wp) * d10 + wp * d11)


class LoopUnrolledOffsetShapeOperator(nn.Module):
    def __init__(self, case_width, path_width, width=256, bands=6,
                 loop_unrolling=True, split_levels=True, eos_density=True, periodic=True):
        super().__init__()
        self.loop_unrolling = loop_unrolling
        self.split_levels = split_levels
        self.eos_density = eos_density
        self.periodic = periodic
        self.register_buffer("bands", 2.0 ** torch.arange(bands, dtype=torch.float32) * math.pi)
        spatial_width = path_width + 2 * bands if periodic else path_width
        self.level = nn.Sequential(nn.Linear(case_width, width), nn.GELU(),
                                   nn.Linear(width, width), nn.GELU(), nn.Linear(width, 10))
        self.film = nn.Sequential(nn.Linear(case_width, width), nn.GELU(),
                                  nn.Linear(width, 2 * width))
        self.spatial = nn.Sequential(nn.Linear(spatial_width, width), nn.GELU(),
                                     nn.Linear(width, width), nn.GELU())
        self.cartesian = nn.Sequential(nn.Linear(3, width), nn.GELU(),
                                       nn.Linear(width, width), nn.GELU())
        self.shape = nn.Sequential(nn.Linear(width + case_width, width), nn.GELU(),
                                   nn.Linear(width, width), nn.GELU(), nn.Linear(width, 7))
        self.absolute = nn.Linear(width + case_width, 8)

    def forward(self, case_embedding, path_features, rotation_angle,
                operating_pressure_pa, eos=None, cartesian_xyz=None):
        case_embedding = case_embedding.reshape(1, -1)
        levels = self.level(case_embedding).squeeze(0)
        expanded_case = case_embedding.expand(path_features.shape[0], -1)
        if self.loop_unrolling:
            if self.periodic:
                phase = path_features[:, :1] * self.bands
                encoded = torch.cat([path_features, torch.sin(phase), torch.cos(phase)], dim=1)
            else:
                encoded = path_features
            spatial = self.spatial(encoded)
        else:
            if cartesian_xyz is None:
                raise ValueError("The no-unrolling ablation requires Cartesian coordinates.")
            spatial = self.cartesian(cartesian_xyz)
        gamma, beta = self.film(expanded_case).chunk(2, dim=1)
        conditioned = spatial * (1 + gamma) + beta
        fused = torch.cat([conditioned, expanded_case], dim=1)
        if not self.split_levels:
            fields = self.absolute(fused)
            temperature_c, pressure_gauge = fields[:, 1], fields[:, 2]
            aligned_u, aligned_v = fields[:, 5], fields[:, 6]
            c, s = torch.cos(rotation_angle), torch.sin(rotation_angle)
            fields = torch.stack([fields[:, 0], temperature_c, pressure_gauge, fields[:, 3],
                                  fields[:, 4], aligned_u * c - aligned_v * s,
                                  aligned_u * s + aligned_v * c, fields[:, 7]], dim=1)
        else:
            shape = self.shape(fused)
            mean = torch.stack([levels[0], levels[1], *([levels.new_zeros(())] * 5)])
            scale = torch.exp(levels[3:10].clamp(-8, 8))
            field_shape = mean[None, :] + scale[None, :] * shape
            aligned_u, aligned_v = field_shape[:, 4], field_shape[:, 5]
            c, s = torch.cos(rotation_angle), torch.sin(rotation_angle)
            fields = torch.stack([levels.new_zeros(len(path_features)), field_shape[:, 0],
                                  field_shape[:, 1], field_shape[:, 2], field_shape[:, 3],
                                  aligned_u * c - aligned_v * s,
                                  aligned_u * s + aligned_v * c, field_shape[:, 6]], dim=1)
        t_k = fields[:, 1] + 273.15
        p_abs = fields[:, 2] + operating_pressure_pa
        if self.eos_density:
            if eos is None:
                raise ValueError("EOS-derived density requires a differentiable CO2 EOS table.")
            density = eos(t_k, p_abs)
            fields[:, 0] = density
        else:
            if self.split_levels:
                fields[:, 0] = self.absolute(fused)[:, 0]
        return fields, levels


def loop_unrolled_huber_loss(prediction, truth, case_level_prediction,
                             case_level_truth, energy_balance_residual,
                             energy_weight=1e-3):
    data_loss = F.huber_loss(prediction, truth, delta=1.0)
    level_loss = F.huber_loss(case_level_prediction, case_level_truth, delta=1.0)
    balance_loss = torch.mean(energy_balance_residual**2)
    return data_loss + level_loss + energy_weight * balance_loss


if not any(spec["name"] == "Loop-Unrolled Offset-Shape Operator" for spec in MODEL_SPECS):
    STATUS_ROWS = [
        row for row in STATUS_ROWS
        if not (
            row.get("model") == "Loop-Unrolled Offset-Shape Operator"
            and row.get("split") == "all_folds"
        )
    ]
    STATUS_ROWS.append({
        "protocol": "kfold",
        "split": "all_folds",
        "model": "Loop-Unrolled Offset-Shape Operator",
        "status": "pending_author_geometry",
        "note": "Training is gated on validated loop centerline, segment, heater/cooler, and rotation metadata.",
    })
    save_progress()


In [ ]:
# Select one fixed model per field by median case nMAE, then export maps for best/median/worst cases.
import matplotlib.pyplot as plt

MODEL_FOR_MAPS = {
    "Mean baseline", "Nearest case", "Random Forest", "Extra Trees",
    "XGBoost", "CatBoost", "DeepONet", "PINN data-only",
    "PINN continuity", "Domain-informed FiLM-MLP",
    "Domain-informed Concat-MLP", "Latent Neural Operator",
    "Transolver-style", "F-FNO", "U-FNO3D", "GAMS-GNN-style",
    "POD-RBF", "POD-GPR", "Loop-Unrolled Offset-Shape Operator",
}
metric_frame = pd.DataFrame(RESULT_ROWS)
oof_metrics = metric_frame.loc[
    (metric_frame["protocol"] == "kfold")
    & metric_frame["model"].isin(MODEL_FOR_MAPS)
].copy()
if oof_metrics.empty:
    raise RuntimeError("No out-of-fold metrics were produced.")

selections = []
for target in TARGETS:
    target_metrics = oof_metrics.loc[oof_metrics["target"] == target]
    model_medians = target_metrics.groupby("model")["mae_over_std"].median().sort_values()
    selected_model = model_medians.index[0]
    ranked = target_metrics.loc[target_metrics["model"] == selected_model].sort_values(
        ["mae_over_std", "case_id"]
    ).reset_index(drop=True)
    for role, row_index in [
        ("best", 0),
        ("median", len(ranked) // 2),
        ("worst", len(ranked) - 1),
    ]:
        row = ranked.iloc[row_index]
        selections.append({
            "target": target,
            "selected_model_by_median_case_nmae": selected_model,
            "case_role": role,
            "case_id": row["case_id"],
            "case_nmae": row["mae_over_std"],
            "case_r2": row["r2"],
        })
selection_frame = pd.DataFrame(selections)
selection_frame.to_csv(OUTPUT_DIR / "case_prediction_selection.csv", index=False)
print(selection_frame.to_string(index=False))

TARGET_GROUPS = {
    "main": ["density_kg_m3", "temperature_c", "pressure_pa"],
    "supplement": [t for t in TARGETS if t not in {"density_kg_m3", "temperature_c", "pressure_pa"}],
}
for group_name, targets_for_plot in TARGET_GROUPS.items():
    fig, axes = plt.subplots(3 * len(targets_for_plot), 3, figsize=(15, 4.0 * len(targets_for_plot)), squeeze=False)
    for row_index, selection in selection_frame.loc[
        selection_frame["target"].isin(targets_for_plot)
    ].reset_index(drop=True).iterrows():
        target = selection["target"]
        case_id = selection["case_id"]
        model_name = selection["selected_model_by_median_case_nmae"]
        target_index = TARGETS.index(target)
        model_file = PRED_DIR / f"{safe(model_name)}__{safe(case_id)}.npz"
        case_file = PRED_DATA_DIR / f"{safe(case_id)}.npz"
        if not model_file.exists() or not case_file.exists():
            raise FileNotFoundError(f"Missing OOF prediction or case data for {model_name}, {case_id}")
        with np.load(model_file) as pred_file, np.load(case_file) as case_file_data:
            prediction = pred_file["prediction"][:, target_index]
            truth = case_file_data["truth"][:, target_index]
            coords = case_file_data["coordinates"]
        rng = np.random.default_rng(SEED + target_index)
        keep = np.arange(len(truth))
        if len(keep) > 25_000:
            keep = np.sort(rng.choice(keep, 25_000, replace=False))
        truth_values, prediction_values = truth[keep], prediction[keep]
        error_values = np.abs(prediction_values - truth_values)
        lo, hi = np.nanpercentile(np.r_[truth_values, prediction_values], [1, 99])
        error_hi = max(float(np.nanpercentile(error_values, 99)), 1e-12)
        axes[row_index, 0].scatter(coords[keep, 0], coords[keep, 1], c=truth_values, s=2, cmap="inferno", vmin=lo, vmax=hi, rasterized=True)
        axes[row_index, 1].scatter(coords[keep, 0], coords[keep, 1], c=prediction_values, s=2, cmap="inferno", vmin=lo, vmax=hi, rasterized=True)
        axes[row_index, 2].scatter(coords[keep, 0], coords[keep, 1], c=error_values, s=2, cmap="magma", vmin=0, vmax=error_hi, rasterized=True)
        for col, title in enumerate(["CFD", model_name, "Absolute error"]):
            axes[row_index, col].set_title(title)
            axes[row_index, col].set_xlabel("X (m)")
            axes[row_index, col].set_ylabel("Y (m)")
            axes[row_index, col].set_aspect("equal", adjustable="datalim")
        axes[row_index, 0].text(
            -0.24, 0.5,
            f"{target}\n{selection['case_role']}\n{case_id}",
            transform=axes[row_index, 0].transAxes,
            ha="right",
            va="center",
            fontsize=8,
        )
    fig.suptitle(
        f"Out-of-fold reconstruction examples: {group_name} targets\n"
        "XY projection; at most 25,000 measured nodes shown per panel",
        y=1.002,
    )
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"prediction_maps_{group_name}_targets.png", dpi=160, bbox_inches="tight")
    fig.savefig(OUTPUT_DIR / f"prediction_maps_{group_name}_targets.pdf", bbox_inches="tight")
    plt.show()
    plt.close(fig)

print("OOF prediction files:", sum(1 for _ in PRED_DIR.glob("*.npz")))
print("Case data files:", sum(1 for _ in PRED_DATA_DIR.glob("*.npz")))
